# Repeat Purchase Prediction, part 4
# Model benchmarking and optimisation

## Notebook purpose

This notebook compares more model families with the logistic regression and histogram gradient boosting from `03_modelling_homework.ipynb`, tunes them, tests blends and weighted ensembles, and picks the model for the final submission. It covers the modelling and evaluation phases of the project.

## Approach

- **Same validation as notebook 3:** four expanding yearly folds (2020–2023) with a 90-day gap, and the holdout from January 2024. Every model is scored on the same rows.
- **Same inputs as notebook 3:** the notebook 2 configuration, including the time-stable rates and the purchase-process probability fitted inside each training part.
- **Defaults first:** each new model runs with its default settings and is compared with the untuned logistic regression and gradient boosting.
- **Then tuning:** each model is tuned on the development folds and compared with its own default, then with logistic regression.
- **One decision rule:** the notebook 2 rule. Log-loss must be clearly lower and F1 not clearly worse, judged with customer-bootstrap intervals and fold consistency. ROC-AUC and balanced accuracy are reported alongside.
- **Holdout only records:** choices are made on development results; the holdout scores the finalists and changes nothing.

## Structure

1. Setup and data
2. Reference models
3. New models with default settings
4. Tuning
5. Scoreboard, blends, weighted ensembles and model choice
6. Threshold and calibration
7. Holdout assessment
8. Final fit and submission
9. Drift and refit check
10. Summary of decisions


# 1. Setup and data

## 1.1. Paths and experiment settings

In [1]:
# Handle paths without computer-specific locations.
from pathlib import Path

PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "data").is_dir() and (folder / "notebooks").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the project. Open the notebook within the repository.")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "preprocessed"
SUBMISSION_DIR = DATA_DIR / "submissions"
OPTUNA_DIR = DATA_DIR / "optuna"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

for folder in (SUBMISSION_DIR, OPTUNA_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# decision: a new experiment version, so 04 searches never mix with the homework ones
EXPERIMENT_VERSION = "project_v1"

N_TRIALS = 60         # completed trials per search
FRESH_SEARCH = False  # True deletes and restarts the named searches of this version

NEEDED = [
    PROCESSED_DIR / "train_clean.parquet",
    PROCESSED_DIR / "test_clean.parquet",
    PROCESSED_DIR / "decision_log_modelling.json",
    PROCESSED_DIR / "column_roles.json",
    PROCESSED_DIR / "preprocessing_config.json",
    RAW_DIR / "sample_submission.csv",
]

missing = [path.name for path in NEEDED if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}. Run 02_preprocessing and 03_modelling_homework first.")

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Experiment version: {EXPERIMENT_VERSION} | trials per search: {N_TRIALS}")

Project root: ml-repeat-purchase
Experiment version: project_v1 | trials per search: 60


## 1.2. Imports, display and plot settings

In [2]:
# Standard library
import sys
import json
import hashlib
import warnings
from time import perf_counter

# Data and numerical calculations
import numpy as np
import pandas as pd

# Charts
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Notebook display
from IPython.display import display

# Hyperparameter search
import optuna
from optuna.importance import FanovaImportanceEvaluator
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice

# Pipelines and custom transformers
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils.validation import check_is_fitted
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Joblib
import joblib
from importlib.metadata import version

# Metrics
from sklearn.metrics import (
    f1_score,
    log_loss,
    roc_auc_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.exceptions import ConvergenceWarning
from scipy.optimize import minimize

# Project modules
project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import src.repeat_purchase_prep as rpp
from src.decision_log import DecisionLog

RANDOM_STATE = rpp.RANDOM_STATE
target = rpp.TARGET

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
# Plot definitions shared by every chart in this notebook
ACCENT = {"burgundy": "#801333", "orange": "#ff8600", "near_black": "#1d212a"}
SERIES_COLOURS = ["#801333", "#f3a481", "#35193e", "#ff8600", "#e13342"]  # one colour per model or group
GRID_COLOUR = "#eeeeee"
FONT = "Inter, Arial, sans-serif"

# Seaborn's rocket palette as Plotly colour scales
ROCKET = [[i / 10, colour] for i, colour in enumerate(sns.color_palette("rocket", 11).as_hex())]
ROCKET_R = [[i / 10, colour] for i, colour in enumerate(sns.color_palette("rocket_r", 11).as_hex())]
# Correlations and loadings: always plotted with zmin=-1, zmax=1
DIVERGING = [[0.0, ACCENT["orange"]], [0.5, "#ffffff"], [1.0, ACCENT["burgundy"]]]

pio.templates["project"] = go.layout.Template(
    layout=dict(
        font=dict(family=FONT, size=13, color=ACCENT["near_black"]),
        title=dict(x=0, xanchor="left", font=dict(size=17)),
        paper_bgcolor="white",
        plot_bgcolor="white",
        colorway=SERIES_COLOURS,
        colorscale=dict(sequential=ROCKET_R, diverging=DIVERGING),
        xaxis=dict(gridcolor=GRID_COLOUR, zeroline=False, linecolor="#cccccc", ticks="outside"),
        yaxis=dict(gridcolor=GRID_COLOUR, zeroline=False, linecolor="#cccccc", ticks="outside"),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(font_family=FONT),
        margin=dict(l=60, r=30, t=70, b=60),
    )
)
pio.templates.default = "project"

# decision: figures saved as HTML (plotly.js from the CDN) and PNG at 300 dpi equivalent
HTML_OPTIONS = dict(include_plotlyjs="cdn")
PNG_OPTIONS = dict(scale=300 / 72)

## 1.3. Data and folds

The cleaned data, column roles and validation settings come from notebook 2. The folds are the four expanding years used in notebook 3, so every model here is scored on the same rows. Only the key checks are repeated.

In [4]:
train = pd.read_parquet(PROCESSED_DIR / "train_clean.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_clean.parquet")
sample_submission = pd.read_csv(RAW_DIR / "sample_submission.csv")

config = json.loads((PROCESSED_DIR / "preprocessing_config.json").read_text(encoding="utf-8"))
roles = json.loads((PROCESSED_DIR / "column_roles.json").read_text(encoding="utf-8"))

# Validation settings saved by notebook 2
GAP_DAYS = int(config["validation"]["gap_days"])
HOLDOUT_START = pd.Timestamp(config["validation"]["holdout_start"])
DEV_END = HOLDOUT_START - pd.Timedelta(days=GAP_DAYS)
VAL_YEARS = list(config["validation"]["validation_years"])

dev = train.loc[train["purchase_date"] < DEV_END].reset_index(drop=True)
holdout = train.loc[train["purchase_date"] >= HOLDOUT_START].reset_index(drop=True)

# Expanding yearly folds: training outcomes close before the validation year starts
folds = []
for year in VAL_YEARS:
    start = pd.Timestamp(year=year, month=1, day=1)
    stop = min(pd.Timestamp(year=year + 1, month=1, day=1), DEV_END)
    train_idx = np.flatnonzero(dev["purchase_date"] < start - pd.Timedelta(days=GAP_DAYS))
    val_idx = np.flatnonzero((dev["purchase_date"] >= start) & (dev["purchase_date"] < stop))
    folds.append((train_idx, val_idx))

# Key checks only; the full set ran in notebook 3
assert set(train.columns) - {target} == set(test.columns)
assert set(sample_submission["ID"]) == set(test["ID"])
replaced = config.get("review_features", {}).get("replaced", [])
assert set(roles["num"]) - {"nbd_repeat_prob"} <= set(dev.columns) and not set(replaced) & set(roles["num"]), \
    "Saved column roles no longer match the data."

log = DecisionLog.load(PROCESSED_DIR / "decision_log_modelling.json")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Development: {len(dev):,} | Holdout: {len(holdout):,}")
print("Fold sizes (train / validation):", {y: (len(tr), len(va)) for y, (tr, va) in zip(VAL_YEARS, folds)})
print(f"Decisions loaded: {len(log)}")

Train: (6469, 55) | Test: (986, 54)
Development: 5,403 | Holdout: 835
Fold sizes (train / validation): {2020: (2578, 670), 2021: (3234, 782), 2022: (4004, 628), 2023: (4705, 536)}
Decisions loaded: 39


## 1.4. Pipeline builders

Two input branches, as decided in notebook 2. The linear branch fills, scales and encodes the columns, and can add the elastic-net selection. The tree branch keeps missing values and codes the categories. Both branches add the purchase-process probability, fitted on each training part from the prior purchase count and tenure. Distance and probability models (KNN, Naive Bayes, LDA) use the linear branch; trees use the tree branch.


In [5]:
# Saved branch decisions from notebook 2
NOISE = list(config["excluded_noise"])
TREE_DROP = list(config["hist_gradient_boosting"]["drop_columns"])
NBD_BLOCK = config.get("review_features", {}).get("fitted_block", {})  # fitted inside each training part
NBD_INPUTS = NBD_BLOCK.get("nbd_repeat_prob", [])


class NBDRepeatProbability(BaseEstimator, TransformerMixin):
    """BG/NBD-style probability of another purchase within the horizon, fitted on the training rows (notebook 2, 6.4)."""

    def __init__(self, horizon=90):
        self.horizon = horizon

    @staticmethod
    def _prob(theta, x, T, h):
        r, alpha, a, b = np.exp(theta)  # log scale keeps the parameters positive
        stop = np.where(x >= 1, a / (a + b + np.maximum(x - 1, 0)), 0.0)
        buy = 1 - np.exp((r + x) * (np.log(alpha + T) - np.log(alpha + T + h)))
        return np.clip((1 - stop) * buy, 1e-6, 1 - 1e-6)

    @staticmethod
    def _inputs(X):
        X = np.asarray(X, dtype=float)
        return np.nan_to_num(X[:, 0]), np.nan_to_num(np.maximum(X[:, 1], 0))

    def fit(self, X, y):
        x, T = self._inputs(X)
        y = np.asarray(y, dtype=float)

        def neg_log_lik(theta):
            p = self._prob(theta, x, T, self.horizon)
            return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

        result = minimize(neg_log_lik, x0=np.log([1.0, 50.0, 1.0, 5.0]), method="Nelder-Mead",
                          options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-9})
        self.theta_ = result.x
        return self

    def transform(self, X):
        x, T = self._inputs(X)
        return self._prob(self.theta_, x, T, self.horizon).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(["nbd_repeat_prob"])


def branch_roles(branch):
    """Numeric, large and small categorical inputs of one branch; the fitted probability is added by the builders."""
    excluded = set(NOISE + (TREE_DROP if branch == "tree" else []) + ["nbd_repeat_prob"])
    return {group: [c for c in roles[group] if c not in excluded] for group in ("num", "high", "low")}


def branch_inputs(branch):
    r = branch_roles(branch)
    columns = r["num"] + r["high"] + r["low"]
    return columns + [c for c in NBD_INPUTS if c not in columns]  # raw inputs of the fitted probability


def build_linear_pipeline(model, select=True):
    """Median fill and scaling, frequency and one-hot encoding, optional elastic-net selection."""
    settings, r = config["linear"], branch_roles("linear")
    blocks = [
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
        ]), r["num"]),
        ("high", Pipeline([("encode", rpp.FrequencyEncoder()), ("scale", StandardScaler())]), r["high"]),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=settings["rare_min_rows"]), r["low"]),
    ]
    if NBD_INPUTS:
        blocks.append(("nbd", Pipeline([("fit", NBDRepeatProbability()), ("scale", StandardScaler())]), NBD_INPUTS))
    steps = [("prep", ColumnTransformer(blocks, sparse_threshold=0))]
    if select:
        steps.append(("select", SelectFromModel(
            LogisticRegression(solver="saga", l1_ratio=settings["selector_l1_ratio"], C=settings["selector_C"],
                               max_iter=5000, random_state=RANDOM_STATE),
            threshold=settings["selector_threshold"],
        )))
    steps.append(("model", clone(model)))
    return Pipeline(steps)


def build_tree_pipeline(model):
    """Small categories as codes (first), large ones as frequencies, numbers untouched; missing values kept."""
    r = branch_roles("tree")
    blocks = [
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), r["low"]),
        ("high", rpp.FrequencyEncoder(), r["high"]),
        ("num", "passthrough", r["num"]),
    ]
    if NBD_INPUTS:
        blocks.append(("nbd", NBDRepeatProbability(), NBD_INPUTS))  # last, so category positions stay first
    prep = ColumnTransformer(blocks, sparse_threshold=0, verbose_feature_names_out=False)
    model = clone(model)
    if isinstance(model, HistGradientBoostingClassifier):
        # decision: native categories and no internal random split, as in notebook 3
        model.set_params(categorical_features=list(range(len(r["low"]))), early_stopping=False)
    return Pipeline([("prep", prep), ("model", model)])


print("Linear inputs:", len(branch_inputs("linear")), "| Tree inputs:", len(branch_inputs("tree")),
      "| fitted probability:", bool(NBD_INPUTS))


Linear inputs: 42 | Tree inputs: 42 | fitted probability: True


## 1.5. Scoring on the folds

Each model is fitted in every fold and scored on the later year. Its threshold is chosen once, on the four years' predictions pooled, with the same rule as notebook 3. The predictions keep the customer and whether that customer appeared in the training part, for the error analysis.

In [6]:
def best_threshold(y, p, tol=0.005):
    """Centre of the longest run of thresholds within tol of the best F1 (grid 0.05 to 0.95)."""
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    actual = np.asarray(y).astype(bool)[:, None]
    predicted = np.asarray(p)[:, None] >= grid
    tp = (predicted & actual).sum(axis=0)
    fp = (predicted & ~actual).sum(axis=0)
    fn = (~predicted & actual).sum(axis=0)
    f1s = 2 * tp / np.maximum(2 * tp + fp + fn, 1)

    near = np.flatnonzero(f1s >= f1s.max() - tol)
    runs = [r for r in np.split(near, np.flatnonzero(np.diff(near) > 1) + 1) if np.any(f1s[r] == f1s.max())]
    chosen = max(runs, key=lambda r: (len(r), -int(r[0])))
    return float(grid[chosen[(len(chosen) - 1) // 2]]), grid, f1s


def f1_floor(y):
    """F1 of predicting a repeat for every row."""
    share = np.mean(y)
    return 2 * share / (1 + share)


def score_folds(pipe, inputs, data=dev):
    """Fit in every fold; return (fold table, out-of-fold predictions, pooled threshold)."""
    rows, oof = [], []
    for (tr, va), year in zip(folds, VAL_YEARS):
        train_part, val_part = data.iloc[tr], data.iloc[va]
        start = perf_counter()
        model = clone(pipe).fit(train_part[inputs], train_part[target])
        seconds = perf_counter() - start

        p = model.predict_proba(val_part[inputs])[:, 1]
        p_train = model.predict_proba(train_part[inputs])[:, 1]
        y = val_part[target].to_numpy()
        steps = model.named_steps
        rows.append({
            "fold": year,
            "n_features": (int(steps["select"].get_support().sum()) if "select" in steps
                           else len(steps["prep"].get_feature_names_out())),
            "log_loss": log_loss(y, p, labels=[0, 1]),
            "train_log_loss": log_loss(train_part[target], p_train, labels=[0, 1]),
            "roc_auc": roc_auc_score(y, p),
            "fit_seconds": seconds,
        })
        # Kept for later error analysis: who the customer is and whether training saw them
        oof.append(pd.DataFrame({
            "fold": year, "ID": val_part["ID"].to_numpy(), "customer_id": val_part["customer_id"].to_numpy(),
            "seen": val_part["customer_id"].isin(train_part["customer_id"]).to_numpy(), "y": y, "p": p,
        }))

    oof = pd.concat(oof, ignore_index=True)
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    table = pd.DataFrame(rows).set_index("fold")
    table["f1"] = [f1_score(part["y"], part["p"] >= threshold) for _, part in oof.groupby("fold")]
    return table, oof, threshold


def summary_row(table, oof, threshold):
    """One line per model: pooled F1 at its own threshold plus fold averages."""
    predicted = oof["p"] >= threshold
    return {
        "n_features": table["n_features"].mean(),
        "f1": f1_score(oof["y"], predicted),
        "floor": f1_floor(oof["y"]),
        "balanced_accuracy": balanced_accuracy_score(oof["y"], predicted),
        "roc_auc": table["roc_auc"].mean(),
        "log_loss": table["log_loss"].mean(),
        "overfit_gap": (table["log_loss"] - table["train_log_loss"]).mean(),
        "threshold": threshold,
        "fit_seconds": table["fit_seconds"].sum(),
    }

## 1.6. Decision rule

Same rule as notebook 2: an option beats the reference only if the 95% customer-bootstrap interval of its log-loss difference sits below zero, its F1 is not clearly worse (the F1 interval does not sit below zero) and its log-loss is higher in at most one fold. It is worse when its log-loss is clearly higher or its F1 clearly lower. Log-loss uses every row, so it detects smaller real differences than F1 at one threshold; F1 stays the reported metric and sets the threshold.


In [7]:
def fast_f1(y, predicted):
    tp = np.sum(y & predicted)
    return 2 * tp / (2 * tp + np.sum(~y & predicted) + np.sum(y & ~predicted))


def row_log_loss(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def bootstrap_diffs(oof_ref, oof_opt, thr_ref, thr_opt, n_boot=2000, seed=RANDOM_STATE):
    """95% intervals of the F1 and log-loss differences (option - reference), resampling whole customers."""
    assert oof_ref["ID"].equals(oof_opt["ID"]), "Options must be scored on the same rows."
    y = oof_ref["y"].to_numpy(dtype=bool)
    pred_ref, pred_opt = oof_ref["p"].to_numpy() >= thr_ref, oof_opt["p"].to_numpy() >= thr_opt
    ll_diff = row_log_loss(y, oof_opt["p"].to_numpy()) - row_log_loss(y, oof_ref["p"].to_numpy())
    groups = list(oof_ref.groupby("customer_id", sort=False).indices.values())

    rng = np.random.default_rng(seed)
    f1_d, ll_d = np.empty(n_boot), np.empty(n_boot)
    for b in range(n_boot):
        rows = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        f1_d[b] = fast_f1(y[rows], pred_opt[rows]) - fast_f1(y[rows], pred_ref[rows])
        ll_d[b] = ll_diff[rows].mean()
    return np.percentile(f1_d, [2.5, 97.5]), np.percentile(ll_d, [2.5, 97.5])


def judge(runs, reference):
    """Scoreboard against the reference: log-loss clearly lower, F1 not clearly worse, at most one worse fold."""
    ref_table, ref_oof, ref_thr = runs[reference]
    ref_ll = row_log_loss(ref_oof["y"].to_numpy(), ref_oof["p"].to_numpy()).mean()
    rows = []
    for name, (table, oof, thr) in runs.items():
        row = {"model": name, **summary_row(table, oof, thr)}
        row["ll_diff"] = row_log_loss(oof["y"].to_numpy(), oof["p"].to_numpy()).mean() - ref_ll
        row["folds_worse"] = int((table["log_loss"] > ref_table["log_loss"]).sum())
        if name == reference:
            row.update(ll_ci_low=np.nan, ll_ci_high=np.nan, ci_low=np.nan, ci_high=np.nan, verdict="reference")
        else:
            (lo, hi), (ll_lo, ll_hi) = bootstrap_diffs(ref_oof, oof, ref_thr, thr)
            verdict = ("better" if ll_hi < 0 and hi >= 0 and row["folds_worse"] <= 1
                       else "worse" if ll_lo > 0 or hi < 0 else "no clear difference")
            row.update(ll_ci_low=ll_lo, ll_ci_high=ll_hi, ci_low=lo, ci_high=hi, verdict=verdict)
        rows.append(row)

    board = pd.DataFrame(rows).set_index("model")
    board.insert(2, "f1_diff", board["f1"] - board.loc[reference, "f1"])
    return board


# 2. Reference models

Three models set the bar for everything that follows. The dummy always predicts the training share of repeats, so its F1 is the "always repeat" floor. The untuned logistic regression and gradient boosting are the starting points of notebook 3. Logistic regression is the reference: a new model has to beat it under the decision rule.

## 2.1. Scoreboard

In [8]:
linear_inputs, tree_inputs = branch_inputs("linear"), branch_inputs("tree")

reference_pipes = {
    # decision: the dummy predicts the training share, so its F1 is the always-repeat floor
    "dummy": (build_linear_pipeline(DummyClassifier(strategy="prior"), select=False), linear_inputs),
    "logistic": (build_linear_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)), linear_inputs),
    "hist_gb": (build_tree_pipeline(HistGradientBoostingClassifier(random_state=RANDOM_STATE)), tree_inputs),
}

reference_runs = {name: score_folds(pipe, inputs) for name, (pipe, inputs) in reference_pipes.items()}

board_reference = judge(reference_runs, reference="logistic")
board_reference.loc["dummy", "n_features"] = 0  # it ignores every column
display(board_reference)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
dummy,0.000,0.709,-0.040,0.709,0.500,0.500,0.689,0.002,0.300,0.475,0.133,4,0.074,0.193,-0.058,-0.018,worse
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.648,0.000,0,NaN,NaN,NaN,NaN,reference
hist_gb,41.000,0.744,-0.004,0.709,0.675,0.774,0.589,0.344,0.350,1.249,0.033,4,0.017,0.050,-0.018,0.010,worse


Logistic regression reaches F1 0.749 against the floor of 0.709, with ROC-AUC 0.785 and balanced accuracy 0.683. Untuned gradient boosting is worse under the rule: its log-loss is 0.589 against 0.558 (interval of the difference [0.017, 0.050]) and higher in all four years, while its F1 of 0.744 is not clearly different. The dummy is clearly worse, as expected, with balanced accuracy 0.500 and ROC-AUC 0.500.


## 2.2. F1 by validation year

In [9]:
fold_f1 = pd.DataFrame({name: reference_runs[name][0]["f1"] for name in ["logistic", "hist_gb"]})
fold_f1["always repeat"] = [f1_floor(part["y"]) for _, part in reference_runs["logistic"][1].groupby("fold")]

fig = go.Figure()
for name in ["logistic", "hist_gb", "always repeat"]:
    fig.add_trace(go.Scatter(
        x=fold_f1.index, y=fold_f1[name], mode="lines+markers" if name != "always repeat" else "lines",
        name=name, line=dict(dash="dash", color="grey") if name == "always repeat" else None,
        hovertemplate="%{x}: F1 %{y:.3f}<extra>" + name + "</extra>",
    ))
fig.update_layout(
    title="Reference models: F1 in each validation year",
    xaxis=dict(title="Validation year", tickmode="array", tickvals=VAL_YEARS),
    yaxis=dict(title="F1 at each model's pooled threshold"),
    height=420,
)
fig.write_html(FIGURES_DIR / "04_reference_f1_by_year.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_reference_f1_by_year.png", **PNG_OPTIONS)
fig.show()

In [10]:
display(fold_f1.round(3))

,logistic,hist_gb,always repeat
fold,,,
2020,0.764,0.753,0.745
2021,0.762,0.751,0.709
2022,0.709,0.706,0.649
2023,0.755,0.768,0.728


Logistic regression stays above the floor in every year, from 0.709 in 2022 to 0.764 in 2020. Its smallest lift is in 2020 (0.764 against 0.745). Gradient boosting beats it only in 2023 (0.768 against 0.755) and is lowest in 2022 (0.706), the year with the lowest floor (0.649).


## Findings

| Model | F1 | ROC-AUC | Balanced accuracy | Log-loss | Verdict against logistic |
|---|---|---|---|---|---|
| Dummy (always repeat) | 0.709 | 0.500 | 0.500 | 0.689 | worse |
| Logistic regression | 0.749 | 0.785 | 0.683 | 0.558 | reference |
| Gradient boosting | 0.744 | 0.774 | 0.675 | 0.589 | worse |

- The untuned logistic regression is the reference for the default models in section 3, at F1 0.749.
- The floor is high (0.709), because about 55% of development purchases are followed by a repeat. Balanced accuracy and ROC-AUC show the gap to a model without information more clearly than F1.
- Gradient boosting's log-loss gap between training and validation (0.344, against 0.008 for logistic regression) shows its default settings follow the training years closely.


# 3. New models with default settings

Each model runs with its library defaults on the same folds, so the comparison shows what each model family brings before any tuning. A short check before each model tests the assumption it relies on.

## 3.1. Naive Bayes

Gaussian Naive Bayes assumes that, within each class, every column is normal and independent of the others. The check measures how far the prepared columns are from that.

In [11]:
# Prepared linear-branch columns of the last fold, the same matrix Naive Bayes sees
tr, _ = folds[-1]
nb_prep = build_linear_pipeline(GaussianNB(), select=False).named_steps["prep"]
X_nb = pd.DataFrame(
    nb_prep.fit_transform(dev.iloc[tr][linear_inputs], dev.iloc[tr][target]),
    columns=nb_prep.get_feature_names_out(),
)
y_nb = dev.iloc[tr][target].to_numpy()

binary = [c for c in X_nb if set(np.unique(X_nb[c])) <= {0.0, 1.0}]
numeric = [c for c in X_nb if c not in binary]
skew_by_class = pd.DataFrame({
    "skew_no_repeat": X_nb.loc[y_nb == 0, numeric].skew(),
    "skew_repeat": X_nb.loc[y_nb == 1, numeric].skew(),
})
skew_by_class["max_abs_skew"] = skew_by_class.abs().max(axis=1)

# Strongest correlation of each numeric column with another, inside the repeat class
corr = X_nb.loc[y_nb == 1, numeric].corr().abs().where(lambda m: ~np.eye(len(m), dtype=bool))
skew_by_class["max_abs_corr"] = corr.max()

print(f"{len(X_nb.columns)} prepared columns: {len(binary)} take only 0 and 1, {len(numeric)} are numeric")
print(f"Numeric columns with |skew| > 1 in either class: {(skew_by_class['max_abs_skew'] > 1).sum()}")
print(f"Numeric columns correlated above 0.8 with another: {(skew_by_class['max_abs_corr'] > 0.8).sum()}")
display(skew_by_class.sort_values("max_abs_skew", ascending=False).head(8).round(2))

73 prepared columns: 39 take only 0 and 1, 34 are numeric
Numeric columns with |skew| > 1 in either class: 30
Numeric columns correlated above 0.8 with another: 16


,skew_no_repeat,skew_repeat,max_abs_skew,max_abs_corr
num__spend_per_year,29.370,42.270,42.270,0.470
num__Total Desc_ Global,31.360,24.280,31.360,0.130
num__Total Portes,15.700,24.020,24.020,0.270
num__purchases_per_year,21.400,7.160,21.400,0.680
num__prior_std_gap_days,5.080,9.020,9.020,0.640
num__Total Desc_ Linha,7.840,6.460,7.840,0.630
num__invoice_count_current_purchase,7.610,5.690,7.610,0.170
num__spend_30d,7.100,3.820,7.100,0.710


Both assumptions fail. Of the 73 prepared columns, 39 take only the values 0 and 1, so they cannot be normal. Of the 34 numeric columns, 30 have a skew above 1 in at least one class (up to 42.3 for `spend_per_year`), and 16 correlate above 0.8 with another column, so the same information is counted several times.


In [12]:
nb_runs = {
    "naive_bayes": score_folds(build_linear_pipeline(GaussianNB(), select=False), linear_inputs),
    "naive_bayes_selected": score_folds(build_linear_pipeline(GaussianNB()), linear_inputs),
}
board_nb = judge({"logistic": reference_runs["logistic"], **nb_runs}, reference="logistic")
display(board_nb)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.648,0.000,0,NaN,NaN,NaN,NaN,reference
naive_bayes,69.250,0.661,-0.088,0.709,0.646,0.727,6.273,2.071,0.070,0.463,5.485,4,4.591,6.450,-0.138,-0.056,worse
naive_bayes_selected,20.250,0.687,-0.062,0.709,0.703,0.754,2.443,0.371,0.050,3.604,1.857,4,1.553,2.197,-0.117,-0.031,worse


On all prepared columns Naive Bayes scores F1 0.661, below the always-repeat floor of 0.709. With the elastic-net columns it reaches 0.687, still below the floor. Both are clearly worse than logistic regression. The probabilities are badly overconfident: log-loss is 6.27 and 2.44 against 0.558, and the selected thresholds sit at 0.07 and 0.05, at the bottom of the grid.


## 3.2. Linear discriminant analysis

LDA fits one normal distribution per class with a shared covariance matrix, and it has to invert that matrix. Strongly or exactly collinear columns make the inversion unstable, so the check measures collinearity.

In [13]:
# Collinearity of the same prepared matrix (last fold), all columns together
X_all = X_nb.to_numpy()
X_all = X_all[:, X_all.std(axis=0) > 0]  # constant columns carry no information
singular_values = np.linalg.svd(X_all - X_all.mean(axis=0), compute_uv=False)
condition_number = singular_values[0] / singular_values[-1]

# Pairs of numeric columns that move almost together
pairs = (
    X_nb[numeric].corr().abs()
    .where(np.triu(np.ones((len(numeric), len(numeric)), dtype=bool), k=1))
    .stack().sort_values(ascending=False)
)

print(f"Condition number of the centred matrix: {condition_number:,.0f}")
print(f"Numeric pairs with |correlation| > 0.9: {(pairs > 0.9).sum()}")
display(pairs.head(6).round(3).rename("abs_correlation").to_frame())

Condition number of the centred matrix: 144,775,749,955,350,720
Numeric pairs with |correlation| > 0.9: 5


,,abs_correlation
num__Total a Pagar,num__Total Liquido,0.996
num__Total Bruto,num__Total Liquido,0.982
num__Total a Pagar,num__Total Bruto,0.972
num__purchase_count_180d,num__purchase_count_365d,0.954
num__purchase_count_90d,num__purchase_count_180d,0.936
num__Total a Pagar,num__Total Imp_,0.894


The condition number is about 1.4 × 10¹⁷, which means the matrix is singular: some columns are exact combinations of others. Two causes: the one-hot encoding keeps every level, so each group of indicators sums to 1, and the accounting identity from the EDA makes `Total a Pagar` equal to `Total Liquido` plus `Total Imp_` (correlation 0.996). Five numeric pairs correlate above 0.9, including the 90-, 180- and 365-day purchase counts.


In [14]:
lda_runs = {
    "lda": score_folds(build_linear_pipeline(LinearDiscriminantAnalysis(), select=False), linear_inputs),
    "lda_selected": score_folds(build_linear_pipeline(LinearDiscriminantAnalysis()), linear_inputs),
}
board_lda = judge({"logistic": reference_runs["logistic"], **lda_runs}, reference="logistic")
display(board_lda)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.648,0.000,0,NaN,NaN,NaN,NaN,reference
lda,69.250,0.739,-0.009,0.709,0.650,0.762,0.591,0.036,0.330,0.584,0.034,4,0.024,0.045,-0.020,0.001,worse
lda_selected,20.250,0.745,-0.003,0.709,0.666,0.778,0.571,0.011,0.340,3.558,0.013,4,0.006,0.020,-0.012,0.005,worse


LDA scores F1 0.739 on all columns and 0.745 on the selected ones, above the floor of 0.709. Both are worse than logistic regression on log-loss (0.591 and 0.571 against 0.558, intervals [0.024, 0.045] and [0.006, 0.020]), and higher in all four years. The default `svd` solver runs despite the singular matrix, because it never inverts the covariance directly.


## 3.3. K-nearest neighbours

KNN predicts from the training rows closest to each new row, so it relies on distances being meaningful. With many columns, the nearest and farthest neighbours can end up almost equally far away. The check compares the two distances on all prepared columns and on the elastic-net selection.

In [15]:
# Distance concentration on the last fold: as columns grow, nearest and farthest neighbours look alike
tr, va = folds[-1]
selected_pipe = build_linear_pipeline(KNeighborsClassifier()).fit(dev.iloc[tr][linear_inputs], dev.iloc[tr][target])

matrices = {
    "all columns": (X_nb.to_numpy(), nb_prep.transform(dev.iloc[va][linear_inputs])),
    "selected columns": tuple(
        selected_pipe.named_steps["select"].transform(selected_pipe.named_steps["prep"].transform(dev.iloc[part][linear_inputs]))
        for part in (tr, va)
    ),
}

rows = []
for name, (X_train_fold, X_val_fold) in matrices.items():
    distances = np.linalg.norm(X_val_fold[:, None, :] - X_train_fold[None, :, :], axis=2)
    nearest, farthest = distances.min(axis=1), distances.max(axis=1)
    rows.append({
        "columns": X_train_fold.shape[1],
        "median_nearest": np.median(nearest),
        "median_farthest": np.median(farthest),
        "median_contrast": np.median((farthest - nearest) / nearest),
    })
display(pd.DataFrame(rows, index=list(matrices)).round(2))

,columns,median_nearest,median_farthest,median_contrast
all columns,73,2.010,61.750,29.670
selected columns,22,0.810,16.720,19.570


The contrast between the farthest and nearest neighbour is higher with all 73 columns (29.7) than with the 22 selected ones (19.6), the opposite of the expected concentration. The farthest distances are driven by the extreme values seen in the Naive Bayes check (skew up to 42), so this measure mostly reflects outliers rather than the number of columns. The selection more than halves the distance to the nearest neighbour (median 0.81 against 2.01).


In [16]:
knn_runs = {
    "knn": score_folds(build_linear_pipeline(KNeighborsClassifier(), select=False), linear_inputs),
    "knn_selected": score_folds(build_linear_pipeline(KNeighborsClassifier()), linear_inputs),
}
board_knn = judge({"logistic": reference_runs["logistic"], **knn_runs}, reference="logistic")
display(board_knn)

print("Distinct probabilities from the selected model:", np.sort(knn_runs["knn_selected"][1]["p"].unique()))

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.648,0.000,0,NaN,NaN,NaN,NaN,reference
knn,69.250,0.721,-0.027,0.709,0.627,0.737,2.200,1.775,0.220,0.575,1.552,4,1.240,1.906,-0.044,-0.012,worse
knn_selected,20.250,0.731,-0.018,0.709,0.644,0.737,2.169,1.745,0.300,3.746,1.557,4,1.235,1.916,-0.033,-0.003,worse


Distinct probabilities from the selected model: [0.  0.2 0.4 0.6 0.8 1. ]


With five neighbours, KNN scores F1 0.721 on all columns and 0.731 on the selected ones, both above the floor of 0.709 and clearly worse than logistic regression. It gives only six probabilities (0, 0.2, … 1), so its threshold of 0.30 means "at least two of the five neighbours repeated". Many predictions at exactly 0 or 1 explain the high log-loss (about 2.2), and the training–validation gap of about 1.75 shows each training row votes for itself.


## 3.4. Decision tree

A tree with default settings keeps splitting until every leaf is pure, so it can memorise the training rows. The check measures how large the default tree grows in each fold and how many leaves hold a single row. Missing values stay in: the tree sends them down whichever branch fits the training rows better.

In [17]:
# Size of the default tree in each fold, fitted on the tree branch with missing values kept
tree_rows = []
for (tr, va), year in zip(folds, VAL_YEARS):
    fitted = build_tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)).fit(
        dev.iloc[tr][tree_inputs], dev.iloc[tr][target]
    )
    tree = fitted.named_steps["model"]
    leaf_sizes = tree.tree_.n_node_samples[tree.tree_.children_left == -1]
    tree_rows.append({
        "fold": year,
        "training_rows": len(tr),
        "depth": tree.get_depth(),
        "leaves": tree.get_n_leaves(),
        "median_leaf_size": np.median(leaf_sizes),
        "single_row_leaves": np.mean(leaf_sizes == 1),
    })
display(pd.DataFrame(tree_rows).set_index("fold").round(3))

,training_rows,depth,leaves,median_leaf_size,single_row_leaves
fold,,,,,
2020,2578,26,443,2.000,0.359
2021,3234,22,568,2.000,0.377
2022,4004,30,685,2.000,0.364
2023,4705,29,790,2.000,0.359


The default tree grows with the data: from depth 26 and 443 leaves in the 2020 fold to depth 29 and 790 leaves in 2023. The median leaf holds two training rows, and 36–38% of the leaves hold a single row, so most answers rest on one or two purchases.


In [18]:
tree_runs = {"decision_tree": score_folds(build_tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)), tree_inputs)}
board_tree = judge({"logistic": reference_runs["logistic"], **tree_runs}, reference="logistic")
display(board_tree)

print("Distinct probabilities:", tree_runs["decision_tree"][1]["p"].nunique())

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.648,0.000,0,NaN,NaN,NaN,NaN,reference
decision_tree,41.000,0.631,-0.118,0.709,0.599,0.597,14.478,14.478,0.500,0.522,13.228,4,11.601,14.763,-0.156,-0.086,worse


Distinct probabilities: 2


The default tree scores F1 0.631, well below the floor of 0.709, with ROC-AUC 0.597 and balanced accuracy 0.599. Its pure leaves give only two probabilities, 0 and 1, so the threshold has no effect and log-loss is extreme (14.5). The training–validation gap equals the validation log-loss, because the tree fits its training rows perfectly.


## 3.5. Random forest

A random forest averages many deep trees, each grown on a bootstrap sample of the rows. The rows left out of each sample give an out-of-bag score at no extra cost, but those rows are drawn at random from the same years. The check compares that score with F1 on the later validation year, to see whether random-row scoring would flatter the model here.

In [19]:
# Out-of-bag F1 (random rows of the training years) against F1 on the later validation year
oob_rows = []
for (tr, va), year in zip(folds, VAL_YEARS):
    forest_pipe = build_tree_pipeline(RandomForestClassifier(oob_score=True, random_state=RANDOM_STATE, n_jobs=-1))
    forest_pipe.fit(dev.iloc[tr][tree_inputs], dev.iloc[tr][target])
    forest = forest_pipe.named_steps["model"]

    p_oob = forest.oob_decision_function_[:, 1]
    p_val = forest_pipe.predict_proba(dev.iloc[va][tree_inputs])[:, 1]
    oob_rows.append({
        "fold": year,
        "oob_f1": f1_score(dev.iloc[tr][target], p_oob >= 0.5),
        "validation_f1": f1_score(dev.iloc[va][target], p_val >= 0.5),
        "median_tree_depth": np.median([t.get_depth() for t in forest.estimators_]),
    })
oob_table = pd.DataFrame(oob_rows).set_index("fold")
oob_table["optimism"] = oob_table["oob_f1"] - oob_table["validation_f1"]
display(oob_table.round(3))

,oob_f1,validation_f1,median_tree_depth,optimism
fold,,,,
2020,0.734,0.713,25.000,0.021
2021,0.735,0.743,25.500,-0.009
2022,0.733,0.706,26.000,0.027
2023,0.735,0.728,27.000,0.007


At a 0.5 cut-off, the out-of-bag F1 differs from the later year's F1 by -0.009 to +0.027, mostly in the forest's favour. Random-row scoring therefore tends to flatter the forest slightly here, which supports keeping the chronological folds as the basis for every decision. The trees grow to a median depth of 25–27, as deep as the single default tree.


In [20]:
forest_runs = {
    "random_forest": score_folds(
        build_tree_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)), tree_inputs
    )
}
board_forest = judge({"logistic": reference_runs["logistic"], **forest_runs}, reference="logistic")
display(board_forest)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.648,0.000,0,NaN,NaN,NaN,NaN,reference
random_forest,41.000,0.745,-0.004,0.709,0.661,0.774,0.588,0.435,0.400,1.000,0.028,3,-0.002,0.064,-0.016,0.009,no clear difference


The default forest scores F1 0.745, close to logistic regression and far above the single tree's 0.631: averaging 100 deep trees removes most of their individual noise. It shows no clear difference against logistic regression, but its log-loss is higher (0.588 against 0.558) in three of four years, and its training–validation gap of 0.435 shows the deep trees still follow the training years closely.


## 3.6. Support vector machine (pending)

## 3.7. Multilayer perceptron (pending)

## 3.8. Default scoreboard

In [21]:
default_runs = {**reference_runs, **nb_runs, **lda_runs, **knn_runs, **tree_runs, **forest_runs}
board_default = judge(default_runs, reference="logistic").sort_values("f1", ascending=False)
board_default.loc["dummy", "n_features"] = 0
display(board_default[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
                       "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
lda_selected,20.250,0.745,-0.003,-0.012,0.005,0.013,0.006,0.020,4,0.666,0.778,0.571,0.340,worse
random_forest,41.000,0.745,-0.004,-0.016,0.009,0.028,-0.002,0.064,3,0.661,0.774,0.588,0.400,no clear difference
hist_gb,41.000,0.744,-0.004,-0.018,0.010,0.033,0.017,0.050,4,0.675,0.774,0.589,0.350,worse
lda,69.250,0.739,-0.009,-0.020,0.001,0.034,0.024,0.045,4,0.650,0.762,0.591,0.330,worse
knn_selected,20.250,0.731,-0.018,-0.033,-0.003,1.557,1.235,1.916,4,0.644,0.737,2.169,0.300,worse
knn,69.250,0.721,-0.027,-0.044,-0.012,1.552,1.240,1.906,4,0.627,0.737,2.200,0.220,worse
dummy,0.000,0.709,-0.040,-0.058,-0.018,0.133,0.074,0.193,4,0.500,0.500,0.689,0.300,worse
naive_bayes_selected,20.250,0.687,-0.062,-0.117,-0.031,1.857,1.553,2.197,4,0.703,0.754,2.443,0.050,worse


In [22]:
order = board_default.index[::-1]  # best at the top of the chart
fig = go.Figure(go.Scatter(
    x=board_default.loc[order, "f1_diff"], y=order, mode="markers",
    marker=dict(size=10, color=ACCENT["burgundy"]),
    error_x=dict(
        type="data", symmetric=False,
        array=(board_default.loc[order, "ci_high"] - board_default.loc[order, "f1_diff"]).fillna(0),
        arrayminus=(board_default.loc[order, "f1_diff"] - board_default.loc[order, "ci_low"]).fillna(0),
        color=ACCENT["near_black"], thickness=1.2,
    ),
    customdata=board_default.loc[order, ["f1", "ci_low", "ci_high"]],
    hovertemplate="%{y}<br>F1 %{customdata[0]:.3f}<br>difference %{x:+.3f}"
                  "<br>95% interval [%{customdata[1]:.3f}, %{customdata[2]:.3f}]<extra></extra>",
))
fig.add_vline(x=0, line=dict(color="grey", dash="dash"))
fig.update_layout(
    title="Default models: F1 difference from logistic regression, 95% customer-bootstrap interval",
    xaxis_title="F1 minus logistic regression's F1",
    height=460,
)
fig.write_html(FIGURES_DIR / "04_default_scoreboard.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_default_scoreboard.png", **PNG_OPTIONS)
fig.show()

No default model beats logistic regression. The random forest is the only one with no clear difference; every other model is worse. LDA on the selected columns, the random forest and gradient boosting cluster at F1 0.744–0.745. Naive Bayes and the single tree fall below the always-repeat floor.


## Findings

| Model | F1 | F1 difference [95% interval] | Log-loss | ROC-AUC | Main issue at defaults |
|---|---|---|---|---|---|
| Logistic regression | 0.749 | reference | 0.558 | 0.785 | — |
| LDA (selected) | 0.745 | -0.003 [-0.012, 0.005] | 0.571 | 0.778 | singular covariance |
| Random forest | 0.745 | -0.004 [-0.016, 0.009] | 0.588 | 0.774 | deep trees, gap 0.435 |
| Gradient boosting | 0.744 | -0.004 [-0.018, 0.010] | 0.589 | 0.774 | gap 0.344 |
| KNN (selected) | 0.731 | -0.018 [-0.033, -0.003] | 2.169 | 0.737 | six probability values |
| Naive Bayes (selected) | 0.687 | -0.062 [-0.117, -0.031] | 2.443 | 0.754 | overconfident probabilities |
| Decision tree | 0.631 | -0.118 [-0.156, -0.086] | 14.478 | 0.597 | one-row leaves, 0/1 probabilities |

- At default settings, logistic regression stays the best model. Each other family fails in a way its check predicted: overconfidence for Naive Bayes, collinearity for LDA, coarse votes for KNN and overfitting for the trees.
- The elastic-net selection helped Naive Bayes, LDA and KNN, so the selected versions go forward.
- Every failure has a matching setting to tune: calibration and smoothing, shrinkage, the number of neighbours, and pruning or depth limits.


# 4. Tuning

Each model family is tuned on the development folds with two Optuna searches of 60 trials each: one steered by log-loss and one by F1. Every trial records both. The preprocessing is fitted once per fold and reused by every trial, because only the model's settings change. The five best distinct trials of each search are rescored and compared with the family's default model under the decision rule.

## 4.0. Search setup

In [23]:
# Fingerprint of the saved inputs: a change in data or configuration starts new searches
fingerprint = hashlib.sha256(EXPERIMENT_VERSION.encode("utf-8"))
for filename in ["train_clean.parquet", "test_clean.parquet", "column_roles.json", "preprocessing_config.json"]:
    fingerprint.update((PROCESSED_DIR / filename).read_bytes())
INPUT_FINGERPRINT = fingerprint.hexdigest()

STORAGE = f"sqlite:///{(OPTUNA_DIR / f'studies_{EXPERIMENT_VERSION}_{INPUT_FINGERPRINT[:12]}.db').as_posix()}"
STUDY_METADATA = {
    "experiment_version": EXPERIMENT_VERSION,
    "input_fingerprint": INPUT_FINGERPRINT,
    "validation_years": [int(year) for year in VAL_YEARS],
    "gap_days": GAP_DAYS,
}
OBJECTIVES = {"log_loss": "minimize", "f1": "maximize"}


class FoldCache:
    """Preprocessing fitted once per fold on the training rows; matrices reused by every trial."""

    def __init__(self, prep, inputs, data=dev):
        self.folds = []
        for (tr, va), year in zip(folds, VAL_YEARS):
            train_part, val_part = data.iloc[tr], data.iloc[va]
            fitted = clone(prep).fit(train_part[inputs], train_part[target])
            self.folds.append({
                "year": year,
                "X_train": np.asarray(fitted.transform(train_part[inputs]), dtype=float),
                "y_train": train_part[target].to_numpy(),
                "X_val": np.asarray(fitted.transform(val_part[inputs]), dtype=float),
                "y_val": val_part[target].to_numpy(),
                "ID": val_part["ID"].to_numpy(),
                "customer_id": val_part["customer_id"].to_numpy(),
                "seen": val_part["customer_id"].isin(train_part["customer_id"]).to_numpy(),
            })


def evaluate_cached(cache, model):
    """Same output as score_folds, on cached matrices."""
    rows, oof = [], []
    for fold in cache.folds:
        start = perf_counter()
        fitted = clone(model).fit(fold["X_train"], fold["y_train"])
        seconds = perf_counter() - start
        p = fitted.predict_proba(fold["X_val"])[:, 1]
        p_train = fitted.predict_proba(fold["X_train"])[:, 1]
        rows.append({
            "fold": fold["year"],
            "n_features": fold["X_train"].shape[1],
            "log_loss": log_loss(fold["y_val"], p, labels=[0, 1]),
            "train_log_loss": log_loss(fold["y_train"], p_train, labels=[0, 1]),
            "roc_auc": roc_auc_score(fold["y_val"], p),
            "fit_seconds": seconds,
        })
        oof.append(pd.DataFrame({key: fold[key] for key in ["ID", "customer_id", "seen"]}
                                | {"fold": fold["year"], "y": fold["y_val"], "p": p}))

    oof = pd.concat(oof, ignore_index=True)[["fold", "ID", "customer_id", "seen", "y", "p"]]
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    table = pd.DataFrame(rows).set_index("fold")
    table["f1"] = [f1_score(part["y"], part["p"] >= threshold) for _, part in oof.groupby("fold")]
    return table, oof, threshold

def pick_cache(cache, params):
    """A search can switch between caches (e.g. all or selected columns) through a function of its settings."""
    return cache(params) if callable(cache) else cache

def tune(cache, make_model, suggest, objective, name, n_trials=N_TRIALS):
    """Run or resume a stored Optuna search steered by log-loss or F1; every trial records both."""
    if FRESH_SEARCH:
        try:
            optuna.delete_study(study_name=name, storage=STORAGE)
        except KeyError:
            pass

    study = optuna.create_study(
        study_name=name, storage=STORAGE, load_if_exists=True, direction=OBJECTIVES[objective],
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),  # same seed, same trials on a rerun
    )
    expected = {**STUDY_METADATA, "objective": objective}
    if study.trials and any(study.user_attrs.get(k) != v for k, v in expected.items()):
        raise ValueError(f"{name}: stored search was run with other settings. Use a new experiment version.")
    for key, value in expected.items():
        study.set_user_attr(key, value)

    def run_trial(trial):
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)  # a model that does not converge stops the search
            params = suggest(trial)
            table, oof, threshold = evaluate_cached(pick_cache(cache, params), make_model(params))
        metrics = summary_row(table, oof, threshold)
        for key in ["f1", "log_loss", "roc_auc", "balanced_accuracy", "threshold", "overfit_gap"]:
            trial.set_user_attr(key, float(metrics[key]))
        return metrics[objective]

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < n_trials:
        study.optimize(run_trial, n_trials=n_trials - done, show_progress_bar=False)
    print(f"{name}: {len(study.trials)} trials")
    return study


def trials_table(study):
    """Completed trials, best first by the steering objective."""
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    table = pd.DataFrame([{"trial": t.number, **t.params, **t.user_attrs} for t in done]).set_index("trial")
    objective = study.user_attrs["objective"]
    return table.sort_values(objective, ascending=OBJECTIVES[objective] == "minimize", kind="stable")


def shortlist(studies, cache, make_model, k=5):
    """Rescore the k best trials of each search, named like 'f1_trial_12'."""
    runs = {}
    for objective, study in studies.items():
        # Trials with identical scores are the same model in practice; keep one of each
        distinct = trials_table(study).round({"f1": 4, "log_loss": 4}).drop_duplicates(subset=["f1", "log_loss"])
        for number in distinct.head(k).index:
            params = study.trials[number].params
            runs[f"{objective}_trial_{number}"] = (evaluate_cached(pick_cache(cache, params), make_model(params)), params)
    return runs


def choose_winner(runs, reference):
    """Keep the reference unless a candidate is better; among better ones, fewer columns then lower log-loss."""
    board = judge(runs, reference)
    better = board.index[board["verdict"] == "better"].tolist()
    if not better:
        return reference, board
    best = board.loc[better].sort_values(["n_features", "ll_diff"], ascending=[True, True]).index[0]
    return best, board

### Tuned references and tracking

The tuned models are compared with logistic regression at its default settings, the reference used throughout section 3, and with gradient boosting at the settings selected in notebook 3 (log-loss trial 59). Notebook 3 also tuned logistic regression (log-loss trial 51); that version is not used here, so the reference stays the same as in section 3. After each family is tuned, its chosen model joins one comparison against logistic regression, shown beside its default to make the effect of tuning visible.


In [24]:
# Tuned references from notebook 3: logistic regression kept its defaults, gradient boosting took F1 trial 44
hgb_params = next(d.carries["tree_params"] for d in log.items if d.step == "D33")
tuned_runs = {
    "logistic": reference_runs["logistic"],
    "hist_gb": score_folds(
        build_tree_pipeline(HistGradientBoostingClassifier(**hgb_params, random_state=RANDOM_STATE)), tree_inputs
    ),
}
# Each family's default run, to show what tuning added
default_of = {"logistic": reference_runs["logistic"], "hist_gb": reference_runs["hist_gb"]}


def plot_search(studies, model_name):
    """Search diagnostics, as in notebook 3: history, setting importance, and every trial's scores."""
    for objective, study in studies.items():
        plot_optimization_history(study).update_layout(
            title=f"{model_name}, steered by {objective}: search history").show()
        plot_param_importances(study, evaluator=FanovaImportanceEvaluator(seed=RANDOM_STATE)).update_layout(
            title=f"{model_name}, steered by {objective}: which settings mattered").show()
    plot_slice(studies["log_loss"]).update_layout(title=f"{model_name}: log-loss against each setting").show()

    # Left: log-loss against F1 for every trial. Right: training against validation log-loss.
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Development performance", "Training and validation loss"])
    for i, (objective, study) in enumerate(studies.items()):
        trials = trials_table(study)
        train_loss = trials["log_loss"] - trials["overfit_gap"]
        style = dict(mode="markers", marker=dict(color=SERIES_COLOURS[i], size=7, opacity=0.7),
                     legendgroup=objective, name=f"steered by {objective}")
        fig.add_trace(go.Scatter(x=trials["log_loss"], y=trials["f1"], **style,
                                 hovertemplate="log-loss %{x:.3f}<br>F1 %{y:.3f}<extra></extra>"), row=1, col=1)
        fig.add_trace(go.Scatter(x=train_loss, y=trials["log_loss"], **style, showlegend=False,
                                 hovertemplate="training %{x:.3f}<br>validation %{y:.3f}<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0.4, 0.8], y=[0.4, 0.8], mode="lines", line=dict(dash="dash", color="grey"),
                             name="equal losses"), row=1, col=2)
    fig.update_xaxes(title_text="Mean validation log-loss", row=1, col=1)
    fig.update_yaxes(title_text="Pooled F1", row=1, col=1)
    fig.update_xaxes(title_text="Mean training log-loss", row=1, col=2)
    fig.update_yaxes(title_text="Mean validation log-loss", row=1, col=2)
    fig.update_layout(title=f"{model_name}: every trial of both searches", height=450)
    fig.show()


def plot_tuning_progress():
    """Tuned families so far against logistic regression, with each family's default for contrast."""
    board = judge(tuned_runs, reference="logistic")
    ref_f1 = board.loc["logistic", "f1"]
    default_diff = pd.Series({name: summary_row(*run)["f1"] - ref_f1 for name, run in default_of.items()})
    order = board.sort_values("f1").index

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=default_diff[order], y=order, mode="markers", name="default",
        marker=dict(symbol="circle-open", size=11, color=ACCENT["near_black"]),
        hovertemplate="%{y} default<br>difference %{x:+.3f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=board.loc[order, "f1_diff"], y=order, mode="markers", name="tuned",
        marker=dict(size=11, color=ACCENT["burgundy"]),
        error_x=dict(type="data", symmetric=False,
                     array=(board.loc[order, "ci_high"] - board.loc[order, "f1_diff"]).fillna(0),
                     arrayminus=(board.loc[order, "f1_diff"] - board.loc[order, "ci_low"]).fillna(0),
                     color=ACCENT["burgundy"], thickness=1.2),
        customdata=board.loc[order, ["f1", "ci_low", "ci_high", "verdict"]],
        hovertemplate="%{y} tuned<br>F1 %{customdata[0]:.3f}<br>difference %{x:+.3f}"
                      "<br>95% interval [%{customdata[1]:.3f}, %{customdata[2]:.3f}]<br>%{customdata[3]}<extra></extra>",
    ))
    fig.add_vline(x=0, line=dict(color="grey", dash="dash"))
    fig.update_layout(
        title=f"Tuned models against logistic regression ({len(board)} families so far)",
        xaxis_title="F1 minus logistic regression's F1 (bars: 95% customer-bootstrap interval)",
        height=150 + 55 * len(board),
    )
    fig.write_html(FIGURES_DIR / "04_tuning_progress.html", **HTML_OPTIONS)
    fig.write_image(FIGURES_DIR / "04_tuning_progress.png", **PNG_OPTIONS)
    fig.show()
    display(board[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "balanced_accuracy",
                   "roc_auc", "log_loss", "threshold", "verdict"]])

## 4.1. Naive Bayes

The default model's main problem was overconfidence. Two settings address it: `var_smoothing`, which adds a share of the largest variance to every variance, and calibration, which relearns the probabilities on held-out parts of the training years (sigmoid or isotonic). The elastic-net columns are kept.

In [25]:
# decision: Naive Bayes keeps the elastic-net selection (better than all columns at defaults)
nb_cache = FoldCache(Pipeline(build_linear_pipeline(GaussianNB()).steps[:-1]), linear_inputs)


def suggest_nb(trial):
    return {
        "var_smoothing": trial.suggest_float("var_smoothing", 1e-12, 1.0, log=True),
        "calibration": trial.suggest_categorical("calibration", ["none", "sigmoid", "isotonic"]),
    }


def make_nb(params):
    model = GaussianNB(var_smoothing=params["var_smoothing"])
    if params["calibration"] == "none":
        return model
    # Calibration learns on 5 internal splits of the training years only
    return CalibratedClassifierCV(model, method=params["calibration"], cv=5)

In [26]:
study_nb = {objective: tune(nb_cache, make_nb, suggest_nb, objective, f"naive_bayes_{objective}")
            for objective in OBJECTIVES}

for objective, study in study_nb.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5)[["var_smoothing", "calibration", "f1", "log_loss", "roc_auc", "threshold"]]
    display(top.style.format({"var_smoothing": "{:.1e}"}, precision=3))

naive_bayes_log_loss: 60 trials
naive_bayes_f1: 60 trials

Steered by log_loss


,var_smoothing,calibration,f1,log_loss,roc_auc,threshold
trial,,,,,,
33,2.8e-04,isotonic,0.725,0.582,0.756,0.360
31,2.5e-04,isotonic,0.725,0.582,0.756,0.360
44,2.4e-04,isotonic,0.725,0.582,0.756,0.360
25,3.2e-04,isotonic,0.725,0.582,0.756,0.360
29,2.4e-04,isotonic,0.726,0.582,0.756,0.360



Steered by f1


,var_smoothing,calibration,f1,log_loss,roc_auc,threshold
trial,,,,,,
47,8.8e-01,isotonic,0.737,0.596,0.759,0.340
26,9.8e-01,isotonic,0.736,0.595,0.760,0.340
55,9.9e-01,isotonic,0.736,0.595,0.759,0.340
32,9.5e-01,isotonic,0.736,0.596,0.759,0.340
45,8.8e-01,isotonic,0.735,0.596,0.759,0.330


In [27]:
plot_search(study_nb, "Naive Bayes")

Both searches settle on isotonic calibration (39 and 40 of 60 trials). The log-loss search keeps negligible smoothing (about 3 × 10⁻⁴) and reaches log-loss 0.582 with F1 0.726. The F1 search adds heavy smoothing (0.88–0.99) and reaches F1 0.735–0.737, at a log-loss of 0.595–0.596. The default had F1 0.687 and log-loss 2.443.


In [28]:
nb_candidates = shortlist(study_nb, nb_cache, make_nb)
runs = {"default": nb_runs["naive_bayes_selected"], **{name: run for name, (run, _) in nb_candidates.items()}}
chosen_nb, board_nb_tuned = choose_winner(runs, reference="default")
nb_params = nb_candidates[chosen_nb][1] if chosen_nb != "default" else {}

display(board_nb_tuned[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_nb} {nb_params}")

,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,
default,0.687,0.000,NaN,NaN,0.000,NaN,NaN,0,2.443,0.754,reference
log_loss_trial_33,0.725,0.039,0.009,0.088,-1.833,-2.167,-1.528,0,0.582,0.756,better
log_loss_trial_44,0.725,0.039,0.009,0.088,-1.833,-2.167,-1.528,0,0.582,0.756,better
log_loss_trial_29,0.726,0.039,0.010,0.088,-1.833,-2.167,-1.528,0,0.582,0.756,better
log_loss_trial_30,0.726,0.039,0.010,0.088,-1.833,-2.167,-1.528,0,0.582,0.755,better
log_loss_trial_59,0.726,0.039,0.010,0.088,-1.832,-2.167,-1.528,0,0.582,0.755,better
f1_trial_47,0.737,0.050,0.016,0.106,-1.818,-2.149,-1.517,0,0.596,0.759,better
f1_trial_26,0.736,0.050,0.015,0.106,-1.819,-2.149,-1.518,0,0.595,0.760,better
f1_trial_55,0.736,0.050,0.015,0.106,-1.820,-2.150,-1.519,0,0.595,0.759,better


Chosen: log_loss_trial_33 {'var_smoothing': 0.0002849413234345257, 'calibration': 'isotonic'}


In [29]:
tuned_runs["naive_bayes"] = runs[chosen_nb]
default_of["naive_bayes"] = nb_runs["naive_bayes_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
hist_gb,41.000,0.748,-0.001,-0.013,0.014,-0.017,-0.028,-0.007,0,0.670,0.790,0.541,0.360,better
naive_bayes,20.250,0.725,-0.023,-0.039,-0.010,0.024,0.010,0.037,4,0.647,0.756,0.582,0.360,worse


All ten candidates pass the rule against the default: log-loss falls from 2.443 to 0.582–0.596 and every F1 interval lies above zero. Log-loss trial 33 is chosen as the lowest log-loss at the same column count: isotonic calibration with smoothing 2.8 × 10⁻⁴, F1 0.725 against 0.687. ROC-AUC barely moves (0.754 to 0.756), because isotonic calibration keeps the order of the predictions and only changes their scale.


In [30]:
# Reliability on the pooled development predictions: default against the best calibrated trial
best_calibrated = board_nb_tuned.drop(index="default").sort_values("f1", ascending=False).index[0]
curves = {"default (uncalibrated)": runs["default"][1], f"calibrated ({best_calibrated})": runs[best_calibrated][1]}

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
                         line=dict(dash="dash", color="grey")))
brier = {}
for name, oof in curves.items():
    observed, predicted = calibration_curve(oof["y"], oof["p"], n_bins=10, strategy="quantile")
    brier[name] = brier_score_loss(oof["y"], oof["p"])
    fig.add_trace(go.Scatter(x=predicted, y=observed, mode="lines+markers", name=name,
                             hovertemplate="predicted %{x:.2f}<br>observed %{y:.2f}<extra>" + name + "</extra>"))
fig.update_layout(
    title="Naive Bayes: predicted against observed repeat share (deciles of the predictions)",
    xaxis=dict(title="Mean predicted probability", range=[0, 1]),
    yaxis=dict(title="Observed repeat share", range=[0, 1]),
    height=480,
)
fig.write_html(FIGURES_DIR / "04_naive_bayes_calibration.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_naive_bayes_calibration.png", **PNG_OPTIONS)
fig.show()

print({name: round(value, 3) for name, value in brier.items()})

{'default (uncalibrated)': 0.305, 'calibrated (f1_trial_47)': 0.202}


Calibration repairs the probabilities: the Brier score falls from 0.305 for the uncalibrated model to 0.202 for the calibrated one. The uncalibrated model pushes most predictions to 0 or 1; after calibration the points follow the diagonal of the reliability chart.


## 4.2. Linear discriminant analysis

The default LDA ran on a singular covariance matrix. Shrinkage pulls the estimated covariance towards a diagonal one, which makes it invertible and more stable. The search tries no shrinkage, the automatic Ledoit–Wolf amount and fixed amounts from 0.0001 to 1, on all prepared columns or on the elastic-net selection.

In [31]:
# Two cached input sets: all prepared columns, or the elastic-net selection
lda_caches = {
    "all": FoldCache(build_linear_pipeline(LinearDiscriminantAnalysis(), select=False).named_steps["prep"], linear_inputs),
    "selected": FoldCache(Pipeline(build_linear_pipeline(LinearDiscriminantAnalysis()).steps[:-1]), linear_inputs),
}


def suggest_lda(trial):
    params = {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "shrinkage_mode": trial.suggest_categorical("shrinkage_mode", ["none", "auto", "value"]),
    }
    if params["shrinkage_mode"] == "value":
        params["shrinkage"] = trial.suggest_float("shrinkage", 1e-4, 1.0, log=True)
    return params


def make_lda(params):
    if params["shrinkage_mode"] == "none":
        return LinearDiscriminantAnalysis(solver="svd")
    # Shrinkage needs a solver that estimates the covariance: lsqr
    shrinkage = "auto" if params["shrinkage_mode"] == "auto" else params["shrinkage"]
    return LinearDiscriminantAnalysis(solver="lsqr", shrinkage=shrinkage)

In [32]:
get_lda_cache = lambda params: lda_caches[params["columns"]]

study_lda = {objective: tune(get_lda_cache, make_lda, suggest_lda, objective, f"lda_{objective}")
             for objective in OBJECTIVES}

for objective, study in study_lda.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5).reindex(
        columns=["columns", "shrinkage_mode", "shrinkage", "f1", "log_loss", "roc_auc", "threshold"]
    )
    display(top.style.format({"shrinkage": "{:.3g}"}, precision=3, na_rep="-"))

lda_log_loss: 60 trials
lda_f1: 60 trials

Steered by log_loss


,columns,shrinkage_mode,shrinkage,f1,log_loss,roc_auc,threshold
trial,,,,,,,
54,selected,value,0.114,0.743,0.565,0.780,0.330
56,selected,value,0.118,0.743,0.565,0.780,0.320
52,selected,value,0.1,0.744,0.565,0.780,0.440
53,selected,value,0.0988,0.744,0.565,0.780,0.440
59,selected,value,0.0906,0.745,0.565,0.780,0.440



Steered by f1


,columns,shrinkage_mode,shrinkage,f1,log_loss,roc_auc,threshold
trial,,,,,,,
10,selected,auto,-,0.748,0.569,0.779,0.320
11,selected,auto,-,0.748,0.569,0.779,0.320
12,selected,auto,-,0.748,0.569,0.779,0.320
13,selected,auto,-,0.748,0.569,0.779,0.320
15,selected,auto,-,0.748,0.569,0.779,0.320


In [33]:
plot_search(study_lda, "LDA")

The two searches find different optima. Log-loss prefers the selected columns with a fixed, moderate shrinkage (about 0.09–0.12), at log-loss 0.565. F1 prefers the automatic Ledoit–Wolf amount on the selected columns (41 of 60 trials), at F1 0.748 and log-loss 0.569.


In [34]:
lda_candidates = shortlist(study_lda, get_lda_cache, make_lda)
runs = {"default": lda_runs["lda_selected"], **{name: run for name, (run, _) in lda_candidates.items()}}
chosen_lda, board_lda_tuned = choose_winner(runs, reference="default")
lda_params = lda_candidates[chosen_lda][1] if chosen_lda != "default" else {}

display(board_lda_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_lda} {lda_params}")

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,,
default,20.250,0.745,0.000,NaN,NaN,0.000,NaN,NaN,0,0.571,0.778,reference
log_loss_trial_54,20.250,0.743,-0.002,-0.008,0.003,-0.006,-0.012,-0.000,0,0.565,0.780,better
log_loss_trial_56,20.250,0.743,-0.002,-0.008,0.005,-0.006,-0.012,-0.000,0,0.565,0.780,better
log_loss_trial_52,20.250,0.744,-0.001,-0.017,0.011,-0.006,-0.012,-0.001,0,0.565,0.780,better
log_loss_trial_59,20.250,0.745,-0.000,-0.017,0.011,-0.006,-0.011,-0.001,0,0.565,0.780,better
log_loss_trial_51,20.250,0.747,0.001,-0.015,0.013,-0.006,-0.011,-0.001,0,0.565,0.780,better
f1_trial_10,20.250,0.748,0.003,-0.002,0.009,-0.002,-0.005,0.000,1,0.569,0.779,no clear difference
f1_trial_5,20.250,0.747,0.002,-0.001,0.006,-0.000,-0.000,-0.000,0,0.571,0.778,better
f1_trial_30,69.250,0.746,0.001,-0.005,0.007,0.003,-0.001,0.007,3,0.573,0.774,no clear difference


Chosen: log_loss_trial_54 {'columns': 'selected', 'shrinkage_mode': 'value', 'shrinkage': 0.11423379054425592}


In [35]:
tuned_runs["lda"] = runs[chosen_lda]
default_of["lda"] = lda_runs["lda_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
hist_gb,41.000,0.748,-0.001,-0.013,0.014,-0.017,-0.028,-0.007,0,0.670,0.790,0.541,0.360,better
naive_bayes,20.250,0.725,-0.023,-0.039,-0.010,0.024,0.010,0.037,4,0.647,0.756,0.582,0.360,worse
lda,20.250,0.743,-0.006,-0.015,0.003,0.007,-0.002,0.015,4,0.659,0.780,0.565,0.330,no clear difference


Seven candidates pass the rule against the default LDA. Log-loss trial 54 is chosen: selected columns with shrinkage 0.114, log-loss 0.565 against 0.571 (interval [-0.012, -0.000]) and F1 0.743 against 0.745. The F1-search trials with automatic shrinkage gain at most +0.003 in F1. Against logistic regression, tuned LDA shows no clear difference.


In [36]:
# F1 and log-loss across the shrinkage values tried (both searches, fixed-value trials only)
tried = pd.concat([trials_table(study) for study in study_lda.values()])
tried = tried.loc[tried["shrinkage_mode"] == "value"].sort_values("shrinkage")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Pooled F1", "Mean validation log-loss"])
for i, (columns, part) in enumerate(tried.groupby("columns")):
    for col, metric in enumerate(["f1", "log_loss"], start=1):
        fig.add_trace(go.Scatter(
            x=part["shrinkage"], y=part[metric], mode="markers", name=f"{columns} columns",
            marker=dict(color=SERIES_COLOURS[i], size=8), legendgroup=columns, showlegend=col == 1,
            hovertemplate="shrinkage %{x:.3g}<br>" + metric + " %{y:.3f}<extra>" + columns + "</extra>",
        ), row=1, col=col)
fig.update_xaxes(type="log", title="Shrinkage (log scale)")
fig.update_layout(title="LDA: effect of shrinkage on F1 and log-loss", height=430)
fig.write_html(FIGURES_DIR / "04_lda_shrinkage.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_lda_shrinkage.png", **PNG_OPTIONS)
fig.show()

F1 barely moves with shrinkage: 0.737–0.747 across the fixed amounts tried. Log-loss shows the trade-off more clearly. On the selected columns it falls from 0.571 with almost no shrinkage to 0.565 around 0.1, then rises to 0.619 at the largest amount tried (0.67), where the covariance moves towards a diagonal one and the model ignores how the columns move together. All columns behave the same way (0.576 to 0.570, then 0.622 at 0.50).


## 4.3. K-nearest neighbours

Four settings are searched: the input columns (all or the elastic-net selection), the number of neighbours k from 1 to 151, uniform or distance-weighted votes, and Manhattan (p = 1) or Euclidean (p = 2) distance. Manhattan distance adds absolute differences, so a single extreme column weighs less than with squared differences.

In [37]:
knn_caches = {
    "all": FoldCache(build_linear_pipeline(KNeighborsClassifier(), select=False).named_steps["prep"], linear_inputs),
    "selected": FoldCache(Pipeline(build_linear_pipeline(KNeighborsClassifier()).steps[:-1]), linear_inputs),
}


def suggest_knn(trial):
    return {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "n_neighbors": trial.suggest_int("n_neighbors", 1, 151, log=True),
        "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
        "p": trial.suggest_categorical("p", [1, 2]),  # 1 Manhattan, 2 Euclidean
    }


def make_knn(params):
    return KNeighborsClassifier(n_neighbors=params["n_neighbors"], weights=params["weights"], p=params["p"])

In [38]:
get_knn_cache = lambda params: knn_caches[params["columns"]]

study_knn = {objective: tune(get_knn_cache, make_knn, suggest_knn, objective, f"knn_{objective}")
             for objective in OBJECTIVES}

for objective, study in study_knn.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5)[["columns", "n_neighbors", "weights", "p", "f1", "log_loss", "roc_auc", "threshold"]])

knn_log_loss: 60 trials
knn_f1: 60 trials

Steered by log_loss


,columns,n_neighbors,weights,p,f1,log_loss,roc_auc,threshold
trial,,,,,,,,
54,selected,85,distance,1,0.743,0.565,0.780,0.360
47,selected,83,distance,1,0.742,0.565,0.780,0.380
14,selected,81,distance,1,0.741,0.565,0.780,0.360
28,selected,80,distance,1,0.739,0.565,0.780,0.350
46,selected,80,distance,1,0.739,0.565,0.780,0.350



Steered by f1


,columns,n_neighbors,weights,p,f1,log_loss,roc_auc,threshold
trial,,,,,,,,
12,selected,113,uniform,1,0.745,0.571,0.775,0.360
41,all,104,uniform,1,0.744,0.570,0.777,0.370
23,selected,71,uniform,1,0.743,0.569,0.776,0.360
45,all,147,distance,1,0.743,0.569,0.779,0.370
51,all,147,distance,1,0.743,0.569,0.779,0.370


In [39]:
plot_search(study_knn, "KNN")

The log-loss search converges on the selected columns, distance-weighted votes and Manhattan distance with about 80–85 neighbours (log-loss 0.565, F1 0.739–0.743). The F1 search prefers 70–150 neighbours, with both uniform and distance votes (F1 0.743–0.745, log-loss 0.569–0.571). ROC-AUC rises from 0.737 at the default to 0.775–0.780.


In [40]:
knn_candidates = shortlist(study_knn, get_knn_cache, make_knn)
runs = {"default": knn_runs["knn_selected"], **{name: run for name, (run, _) in knn_candidates.items()}}
chosen_knn, board_knn_tuned = choose_winner(runs, reference="default")
knn_params = knn_candidates[chosen_knn][1] if chosen_knn != "default" else {}

display(board_knn_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_knn} {knn_params}")

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,,
default,20.250,0.731,0.000,NaN,NaN,0.000,NaN,NaN,0,2.169,0.737,reference
log_loss_trial_54,20.250,0.743,0.012,-0.002,0.026,-1.551,-1.922,-1.222,0,0.565,0.780,better
log_loss_trial_47,20.250,0.742,0.011,-0.005,0.025,-1.551,-1.922,-1.222,0,0.565,0.780,better
log_loss_trial_14,20.250,0.741,0.011,-0.004,0.025,-1.551,-1.922,-1.222,0,0.565,0.780,better
log_loss_trial_28,20.250,0.739,0.009,-0.005,0.022,-1.551,-1.922,-1.222,0,0.565,0.780,better
log_loss_trial_21,20.250,0.742,0.011,-0.002,0.024,-1.551,-1.921,-1.222,0,0.565,0.780,better
f1_trial_12,20.250,0.745,0.015,0.001,0.029,-1.545,-1.919,-1.215,0,0.571,0.775,better
f1_trial_41,69.250,0.744,0.014,0.000,0.029,-1.546,-1.918,-1.217,0,0.570,0.777,better
f1_trial_23,20.250,0.743,0.012,-0.002,0.026,-1.547,-1.918,-1.218,0,0.569,0.776,better


Chosen: log_loss_trial_54 {'columns': 'selected', 'n_neighbors': 85, 'weights': 'distance', 'p': 1}


In [41]:
tuned_runs["knn"] = runs[chosen_knn]
default_of["knn"] = knn_runs["knn_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
hist_gb,41.000,0.748,-0.001,-0.013,0.014,-0.017,-0.028,-0.007,0,0.670,0.790,0.541,0.360,better
naive_bayes,20.250,0.725,-0.023,-0.039,-0.010,0.024,0.010,0.037,4,0.647,0.756,0.582,0.360,worse
lda,20.250,0.743,-0.006,-0.015,0.003,0.007,-0.002,0.015,4,0.659,0.780,0.565,0.330,no clear difference
knn,20.250,0.743,-0.006,-0.019,0.006,0.007,-0.015,0.031,3,0.672,0.780,0.565,0.360,no clear difference


All ten candidates pass the rule against the default: log-loss falls from 2.169 to 0.565–0.571, because many neighbours give smooth probabilities instead of six coarse ones. Log-loss trial 54 is chosen as the lowest log-loss at the selected column count: 85 neighbours, distance-weighted votes and Manhattan distance, F1 0.743 against 0.731.


In [42]:
# Elbow: F1 against k for both weightings, columns and distance of the chosen trial held fixed
elbow_cache = knn_caches[knn_params["columns"]]
elbow_rows = []
for weights in ["uniform", "distance"]:
    for k in [1, 3, 5, 9, 15, 25, 41, 61, 81, 110, 151]:
        model = KNeighborsClassifier(n_neighbors=k, weights=weights, p=knn_params["p"])
        table, oof, threshold = evaluate_cached(elbow_cache, model)
        # Training F1 in each fold, at the same pooled threshold
        train_f1 = np.mean([
            f1_score(fold["y_train"], clone(model).fit(fold["X_train"], fold["y_train"])
                     .predict_proba(fold["X_train"])[:, 1] >= threshold)
            for fold in elbow_cache.folds
        ])
        elbow_rows.append({"weights": weights, "k": k, "train_f1": train_f1,
                           "validation_f1": f1_score(oof["y"], oof["p"] >= threshold)})
elbow = pd.DataFrame(elbow_rows)

fig = go.Figure()
for i, (weights, part) in enumerate(elbow.groupby("weights", sort=False)):
    fig.add_trace(go.Scatter(x=part["k"], y=part["validation_f1"], mode="lines+markers",
                             name=f"validation, {weights}", line=dict(color=SERIES_COLOURS[i]),
                             hovertemplate="k = %{x}<br>F1 %{y:.3f}<extra>validation, " + weights + "</extra>"))
    fig.add_trace(go.Scatter(x=part["k"], y=part["train_f1"], mode="lines", name=f"training, {weights}",
                             line=dict(color=SERIES_COLOURS[i], dash="dot"),
                             hovertemplate="k = %{x}<br>F1 %{y:.3f}<extra>training, " + weights + "</extra>"))
fig.add_hline(y=f1_floor(dev[target]), line=dict(dash="dash", color="grey"),
              annotation_text="always repeat", annotation_position="bottom right")
fig.update_layout(
    title=f"KNN elbow: F1 against the number of neighbours (p = {knn_params['p']}, {knn_params['columns']} columns)",
    xaxis=dict(title="Number of neighbours k (log scale)", type="log", tickvals=sorted(elbow["k"].unique())),
    yaxis_title="F1",
    height=470,
)
fig.write_html(FIGURES_DIR / "04_knn_elbow.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_knn_elbow.png", **PNG_OPTIONS)
fig.show()

display(elbow.pivot(index="k", columns="weights", values=["train_f1", "validation_f1"]).round(3))

train_f1         validation_f1        
weights distance uniform      distance uniform
k                                             
1          1.000   1.000         0.674   0.674
3          1.000   0.808         0.731   0.730
5          1.000   0.807         0.741   0.739
9          1.000   0.780         0.735   0.737
15         1.000   0.770         0.736   0.736
25         1.000   0.763         0.740   0.737
41         1.000   0.760         0.740   0.737
61         1.000   0.755         0.740   0.741
81         1.000   0.753         0.741   0.740
110        1.000   0.751         0.746   0.743
151        1.000   0.748         0.741   0.742

With uniform votes the classic elbow appears: training F1 falls from 1.000 at k = 1 to 0.748 at 151, while validation F1 rises from 0.674 to 0.737 by k = 25 and then stays between 0.737 and 0.743. The elbow sits around k = 25; larger k adds at most +0.006. With distance weighting, training F1 is 1.000 at every k, because each training row is its own nearest neighbour at distance zero, so only its validation curve is informative. It peaks at 0.746 at k = 110.


## 4.4. Decision tree

The default tree grew until almost every leaf was pure and fell below the floor. Three settings limit its growth: `max_depth` caps the number of questions, `min_samples_leaf` sets the smallest group that can answer, and `ccp_alpha` removes branches whose impurity reduction is worth less than alpha per leaf (cost-complexity pruning). Before the search, the pruning path and a depth curve show how size and performance move together.

### Pruning path

In [43]:
# Missing values stay in the tree branch; preprocessing is cached once per fold
tree_cache = FoldCache(build_tree_pipeline(DecisionTreeClassifier()).named_steps["prep"], tree_inputs)

# Cost-complexity pruning path of the full tree in the last fold: the alphas at which branches disappear
last = tree_cache.folds[-1]
path = DecisionTreeClassifier(random_state=RANDOM_STATE).cost_complexity_pruning_path(last["X_train"], last["y_train"])
# Log-spaced, so small and large prunings are both visible
alphas = np.geomspace(1e-5, path.ccp_alphas[:-1].max(), 30)

path_rows = []
for alpha in alphas:
    table, oof, threshold = evaluate_cached(tree_cache, DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_STATE))
    leaves = DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_STATE).fit(
        last["X_train"], last["y_train"]).get_n_leaves()
    path_rows.append({"ccp_alpha": alpha, "leaves_last_fold": leaves,
                      "validation_f1": f1_score(oof["y"], oof["p"] >= threshold),
                      "log_loss": table["log_loss"].mean()})
pruning = pd.DataFrame(path_rows)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=pruning["ccp_alpha"], y=pruning["leaves_last_fold"], mode="lines+markers",
                         name="leaves (last fold)", line=dict(color=ACCENT["near_black"]),
                         hovertemplate="alpha %{x:.2e}<br>%{y} leaves<extra></extra>"), secondary_y=False)
fig.add_trace(go.Scatter(x=pruning["ccp_alpha"], y=pruning["validation_f1"], mode="lines+markers",
                         name="validation F1 (pooled)", line=dict(color=ACCENT["burgundy"]),
                         hovertemplate="alpha %{x:.2e}<br>F1 %{y:.3f}<extra></extra>"), secondary_y=True)
fig.add_hline(y=f1_floor(dev[target]), line=dict(dash="dash", color="grey"), secondary_y=True,
              annotation_text="always repeat", annotation_position="bottom right")
fig.update_xaxes(type="log", title="ccp_alpha (log scale)")
fig.update_yaxes(title_text="Leaves", type="log", secondary_y=False)
fig.update_yaxes(title_text="F1", secondary_y=True)
fig.update_layout(title="Decision tree: pruning path, tree size and validation F1", height=450)
fig.write_html(FIGURES_DIR / "04_tree_pruning_path.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_tree_pruning_path.png", **PNG_OPTIONS)
fig.show()

best_alpha = pruning.loc[pruning["validation_f1"].idxmax()]
print(f"Path alphas: {path.ccp_alphas.min():.1e} to {path.ccp_alphas[:-1].max():.1e}")
print(f"Best on the path: alpha {best_alpha['ccp_alpha']:.1e}, {best_alpha['leaves_last_fold']:.0f} leaves, "
      f"F1 {best_alpha['validation_f1']:.3f}")

Path alphas: 0.0e+00 to 1.1e-02
Best on the path: alpha 3.3e-03, 7 leaves, F1 0.741


The pruning path of the last fold runs from alpha 0 to about 1.1 × 10⁻². Small alphas leave the hundreds of one-row leaves in place. The best point on the path is alpha 3.3 × 10⁻³, a tree of only 7 leaves with F1 0.741: most of the gain comes from removing the small leaves.


### Depth curve

In [44]:
# Training against validation F1 as the tree is allowed to grow deeper
depth_rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    table, oof, threshold = evaluate_cached(tree_cache, model)
    train_f1 = np.mean([
        f1_score(fold["y_train"], clone(model).fit(fold["X_train"], fold["y_train"])
                 .predict_proba(fold["X_train"])[:, 1] >= threshold)
        for fold in tree_cache.folds
    ])
    depth_rows.append({"max_depth": "none" if depth is None else str(depth), "train_f1": train_f1,
                       "validation_f1": f1_score(oof["y"], oof["p"] >= threshold)})
depth_curve = pd.DataFrame(depth_rows).set_index("max_depth")

fig = go.Figure()
for column, name in [("train_f1", "training (mean of folds)"), ("validation_f1", "validation (pooled)")]:
    fig.add_trace(go.Scatter(x=depth_curve.index, y=depth_curve[column], mode="lines+markers", name=name,
                             hovertemplate="max_depth %{x}<br>F1 %{y:.3f}<extra>" + name + "</extra>"))
fig.add_hline(y=f1_floor(dev[target]), line=dict(dash="dash", color="grey"),
              annotation_text="always repeat", annotation_position="bottom right")
fig.update_layout(title="Decision tree: training and validation F1 against maximum depth",
                  xaxis_title="max_depth", yaxis_title="F1", height=430)
fig.write_html(FIGURES_DIR / "04_tree_depth_curve.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_tree_depth_curve.png", **PNG_OPTIONS)
fig.show()

display(depth_curve.round(3).T)

max_depth,1,2,3,4,5,6,8,10,12,15,20,none
train_f1,0.680,0.726,0.720,0.753,0.764,0.782,0.775,0.810,0.889,0.959,0.992,1.000
validation_f1,0.727,0.737,0.730,0.732,0.723,0.714,0.694,0.674,0.661,0.638,0.635,0.631


Training F1 climbs from 0.680 at depth 1 to 1.000 for the unlimited tree, while validation F1 peaks at 0.737 at depth 2 and falls steadily after depth 4, to 0.631. From depth 5 on, the two curves move apart: each extra level fits the training years more closely and the later year less well.


### Search

In [45]:
def suggest_tree(trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 2, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 300, log=True),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-6, 0.05, log=True),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "class_weight": trial.suggest_categorical("class_weight", ["none", "balanced"]),
    }


def make_tree(params):
    settings = {**params, "class_weight": None if params["class_weight"] == "none" else "balanced"}
    return DecisionTreeClassifier(**settings, random_state=RANDOM_STATE)


study_tree = {objective: tune(tree_cache, make_tree, suggest_tree, objective, f"decision_tree_{objective}")
              for objective in OBJECTIVES}

for objective, study in study_tree.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5)[["max_depth", "min_samples_leaf", "ccp_alpha", "criterion", "class_weight",
                                       "f1", "log_loss", "roc_auc", "threshold"]]
    display(top.style.format({"ccp_alpha": "{:.1e}"}, precision=3))

decision_tree_log_loss: 60 trials
decision_tree_f1: 60 trials

Steered by log_loss


,max_depth,min_samples_leaf,ccp_alpha,criterion,class_weight,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
32,2,28,1.9e-04,entropy,balanced,0.742,0.557,0.759,0.310
34,2,19,3.4e-05,entropy,balanced,0.742,0.557,0.759,0.310
36,2,88,1.7e-05,entropy,balanced,0.742,0.557,0.759,0.310
38,2,190,4.4e-05,entropy,balanced,0.742,0.557,0.759,0.310
39,2,99,1.3e-05,entropy,balanced,0.742,0.557,0.759,0.310



Steered by f1


,max_depth,min_samples_leaf,ccp_alpha,criterion,class_weight,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
58,17,70,3.9e-03,entropy,none,0.746,0.561,0.772,0.380
33,12,32,4.2e-03,entropy,none,0.745,0.561,0.772,0.370
40,13,38,2.1e-03,entropy,none,0.744,0.598,0.763,0.420
47,6,25,4.2e-03,entropy,none,0.744,0.649,0.771,0.370
24,20,53,2.5e-04,entropy,none,0.744,0.606,0.761,0.370


In [46]:
plot_search(study_tree, "Decision tree")

The two searches find different trees. The log-loss search settles on stumps of depth 2 with entropy and balanced class weights (log-loss 0.557, F1 0.742). The F1 search keeps deeper trees (depth 6–20) with large leaves (25–70 rows), alpha between 3 × 10⁻⁴ and 4 × 10⁻³, entropy and no class weights, reaching F1 0.744–0.746 at log-loss 0.561–0.649.


In [47]:
tree_candidates = shortlist(study_tree, tree_cache, make_tree)
runs = {"default": tree_runs["decision_tree"], **{name: run for name, (run, _) in tree_candidates.items()}}
chosen_tree, board_tree_tuned = choose_winner(runs, reference="default")
tree_params = tree_candidates[chosen_tree][1] if chosen_tree != "default" else {}

display(board_tree_tuned[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_tree} {tree_params}")

# Size of the chosen tree in the last fold
chosen_fit = make_tree(tree_params).fit(last["X_train"], last["y_train"])
print(f"Chosen tree in the last fold: depth {chosen_fit.get_depth()}, {chosen_fit.get_n_leaves()} leaves")

,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,
default,0.631,0.000,NaN,NaN,0.000,NaN,NaN,0,14.478,0.597,reference
log_loss_trial_32,0.742,0.111,0.079,0.151,-13.229,-14.773,-11.596,0,0.557,0.759,better
log_loss_trial_46,0.742,0.111,0.079,0.151,-13.226,-14.773,-11.592,0,0.560,0.759,better
log_loss_trial_12,0.733,0.103,0.072,0.141,-13.224,-14.767,-11.586,0,0.561,0.764,better
log_loss_trial_22,0.737,0.106,0.076,0.141,-13.224,-14.770,-11.590,0,0.562,0.764,better
log_loss_trial_7,0.736,0.105,0.075,0.143,-13.222,-14.763,-11.589,0,0.563,0.761,better
f1_trial_58,0.746,0.115,0.082,0.156,-13.223,-14.763,-11.591,0,0.561,0.772,better
f1_trial_33,0.745,0.114,0.081,0.155,-13.223,-14.762,-11.589,0,0.561,0.772,better
f1_trial_40,0.744,0.114,0.082,0.152,-13.188,-14.727,-11.542,0,0.598,0.763,better


Chosen: log_loss_trial_32 {'max_depth': 2, 'min_samples_leaf': 28, 'ccp_alpha': 0.00018917343207013604, 'criterion': 'entropy', 'class_weight': 'balanced'}
Chosen tree in the last fold: depth 2, 4 leaves


All ten candidates are clearly better than the default tree: F1 gains of +0.103 to +0.115, with every interval well above zero, and log-loss from 14.5 to 0.557–0.649. Log-loss trial 32 is chosen as the lowest log-loss: depth 2, at least 28 rows per leaf, entropy and balanced weights. In the last fold it has 4 leaves, against 790 for the default.


In [48]:
tuned_runs["decision_tree"] = runs[chosen_tree]
default_of["decision_tree"] = tree_runs["decision_tree"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
hist_gb,41.000,0.748,-0.001,-0.013,0.014,-0.017,-0.028,-0.007,0,0.670,0.790,0.541,0.360,better
naive_bayes,20.250,0.725,-0.023,-0.039,-0.010,0.024,0.010,0.037,4,0.647,0.756,0.582,0.360,worse
lda,20.250,0.743,-0.006,-0.015,0.003,0.007,-0.002,0.015,4,0.659,0.780,0.565,0.330,no clear difference
knn,20.250,0.743,-0.006,-0.019,0.006,0.007,-0.015,0.031,3,0.672,0.780,0.565,0.360,no clear difference
decision_tree,41.000,0.742,-0.007,-0.020,0.008,-0.001,-0.016,0.014,2,0.649,0.759,0.557,0.310,no clear difference


Tuning moves the tree from F1 0.631 to 0.742, the largest gain of any family. Its log-loss (0.557) now matches logistic regression's, so it shows no clear difference. After five families, gradient boosting from notebook 3 is clearly better than logistic regression (log-loss 0.541, interval [-0.028, -0.007]); LDA, KNN and the tree show no clear difference, and Naive Bayes stays worse.


## 4.5. Random forest

The default forest grew 100 unlimited trees with √p columns per split, and it followed the training years closely (log-loss gap 0.435). The search varies the number of trees, the share of columns tried at each split, the minimum leaf size, the depth limit and the class weights. Averaging more trees smooths the probabilities, while fewer columns per split make the trees more different from one another.


In [49]:
# Same cached tree-branch matrices as the single tree
forest_cache = tree_cache


def suggest_forest(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_features": trial.suggest_float("max_features", 0.1, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 100, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "class_weight": trial.suggest_categorical("class_weight", ["none", "balanced", "balanced_subsample"]),
    }


def make_forest(params):
    settings = {**params, "class_weight": None if params["class_weight"] == "none" else params["class_weight"]}
    return RandomForestClassifier(**settings, random_state=RANDOM_STATE, n_jobs=-1)


study_forest = {objective: tune(forest_cache, make_forest, suggest_forest, objective, f"random_forest_{objective}")
                for objective in OBJECTIVES}

for objective, study in study_forest.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5)[["n_estimators", "max_features", "min_samples_leaf", "max_depth",
                                         "class_weight", "f1", "log_loss", "roc_auc", "threshold"]])

random_forest_log_loss: 60 trials
random_forest_f1: 60 trials

Steered by log_loss


,n_estimators,max_features,min_samples_leaf,max_depth,class_weight,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
58,150,0.661,59,21,balanced_subsample,0.751,0.543,0.790,0.380
33,300,0.656,61,17,balanced_subsample,0.750,0.544,0.789,0.390
57,200,0.663,56,22,balanced_subsample,0.746,0.544,0.789,0.370
35,250,0.617,34,15,balanced_subsample,0.751,0.544,0.790,0.380
42,250,0.686,39,17,balanced_subsample,0.753,0.544,0.790,0.350



Steered by f1


,n_estimators,max_features,min_samples_leaf,max_depth,class_weight,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
14,150,0.156,1,11,balanced_subsample,0.755,0.565,0.782,0.410
57,350,0.123,11,12,balanced_subsample,0.754,0.548,0.788,0.390
4,350,0.142,13,7,balanced_subsample,0.754,0.547,0.788,0.370
8,100,0.276,1,12,balanced_subsample,0.753,0.570,0.777,0.390
16,350,0.339,9,17,balanced_subsample,0.753,0.550,0.786,0.390


Every top trial of both searches uses `balanced_subsample` class weights. The log-loss search prefers many columns per split (`max_features` 0.62–0.69), large leaves (34–61 rows) and 150–300 trees, reaching log-loss 0.543–0.544 and ROC-AUC 0.789–0.790. The F1 search prefers few columns per split (0.12–0.34) and reaches F1 0.753–0.755, at a log-loss of 0.547–0.570.


In [50]:
plot_search(study_forest, "Random forest")

In [51]:
forest_candidates = shortlist(study_forest, forest_cache, make_forest)
runs = {"default": forest_runs["random_forest"], **{name: run for name, (run, _) in forest_candidates.items()}}
chosen_forest, board_forest_tuned = choose_winner(runs, reference="default")
forest_params = forest_candidates[chosen_forest][1] if chosen_forest != "default" else {}

display(board_forest_tuned[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_forest} {forest_params}")

,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,
default,0.745,0.000,NaN,NaN,0.000,NaN,NaN,0,0.588,0.774,reference
log_loss_trial_58,0.751,0.006,-0.005,0.017,-0.042,-0.077,-0.015,0,0.543,0.790,better
log_loss_trial_33,0.750,0.005,-0.006,0.015,-0.041,-0.076,-0.015,0,0.544,0.789,better
log_loss_trial_57,0.746,0.002,-0.009,0.012,-0.041,-0.076,-0.014,0,0.544,0.789,better
log_loss_trial_35,0.751,0.007,-0.003,0.016,-0.041,-0.076,-0.015,0,0.544,0.790,better
log_loss_trial_42,0.753,0.008,-0.001,0.018,-0.041,-0.076,-0.015,0,0.544,0.790,better
f1_trial_14,0.755,0.010,0.001,0.019,-0.022,-0.053,-0.006,0,0.565,0.782,better
f1_trial_57,0.754,0.010,0.001,0.019,-0.038,-0.072,-0.012,0,0.548,0.788,better
f1_trial_4,0.754,0.009,-0.000,0.019,-0.038,-0.072,-0.012,0,0.547,0.788,better


Chosen: log_loss_trial_58 {'n_estimators': 150, 'max_features': 0.6612253350329873, 'min_samples_leaf': 59, 'max_depth': 21, 'class_weight': 'balanced_subsample'}


All ten candidates pass the rule against the default forest, with log-loss falling from 0.588 to 0.543–0.570. Log-loss trial 58 is chosen as the lowest log-loss: 150 trees, 66% of the columns per split, at least 59 rows per leaf, depth up to 21 and `balanced_subsample` weights. It scores F1 0.751 and log-loss 0.543 against 0.745 and 0.588 for the default (interval [-0.077, -0.015]), and is worse in no year.


In [52]:
# Number of trees: the chosen settings with more or fewer trees
settings = forest_params or {"max_features": "sqrt", "min_samples_leaf": 1, "max_depth": None, "class_weight": "none"}
tree_count_rows = []
for n_trees in [10, 25, 50, 100, 200, 300, 500]:
    table, oof, threshold = evaluate_cached(forest_cache, make_forest({**settings, "n_estimators": n_trees}))
    tree_count_rows.append({"n_estimators": n_trees, "validation_f1": f1_score(oof["y"], oof["p"] >= threshold),
                            "log_loss": table["log_loss"].mean(), "roc_auc": table["roc_auc"].mean()})
tree_counts = pd.DataFrame(tree_count_rows).set_index("n_estimators")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Pooled validation F1", "Mean validation log-loss"])
for col, metric in enumerate(["validation_f1", "log_loss"], start=1):
    fig.add_trace(go.Scatter(x=tree_counts.index, y=tree_counts[metric], mode="lines+markers", showlegend=False,
                             line=dict(color=ACCENT["burgundy"]),
                             hovertemplate="%{x} trees<br>%{y:.3f}<extra></extra>"), row=1, col=col)
fig.update_xaxes(type="log", title="Number of trees (log scale)", tickvals=list(tree_counts.index))
fig.update_layout(title="Random forest: effect of the number of trees (chosen settings)", height=420)
fig.write_html(FIGURES_DIR / "04_forest_tree_count.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_forest_tree_count.png", **PNG_OPTIONS)
fig.show()

display(tree_counts.round(3).T)

n_estimators,10,25,50,100,200,300,500
validation_f1,0.748,0.750,0.749,0.747,0.749,0.751,0.749
log_loss,0.559,0.545,0.544,0.544,0.543,0.544,0.544
roc_auc,0.787,0.790,0.790,0.790,0.790,0.790,0.789


F1 is almost flat in the number of trees: 0.747–0.751 from 10 to 500 trees. Log-loss improves from 0.559 with 10 trees to 0.545 with 25 and stays at 0.543–0.544 from 50 onwards; ROC-AUC follows (0.787 to 0.790). More trees mostly smooth the probabilities; beyond 50 they only add fitting time.


In [53]:
tuned_runs["random_forest"] = runs[chosen_forest]
default_of["random_forest"] = forest_runs["random_forest"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
hist_gb,41.000,0.748,-0.001,-0.013,0.014,-0.017,-0.028,-0.007,0,0.670,0.790,0.541,0.360,better
naive_bayes,20.250,0.725,-0.023,-0.039,-0.010,0.024,0.010,0.037,4,0.647,0.756,0.582,0.360,worse
lda,20.250,0.743,-0.006,-0.015,0.003,0.007,-0.002,0.015,4,0.659,0.780,0.565,0.330,no clear difference
knn,20.250,0.743,-0.006,-0.019,0.006,0.007,-0.015,0.031,3,0.672,0.780,0.565,0.360,no clear difference
decision_tree,41.000,0.742,-0.007,-0.020,0.008,-0.001,-0.016,0.014,2,0.649,0.759,0.557,0.310,no clear difference
random_forest,41.000,0.751,0.002,-0.009,0.015,-0.014,-0.027,-0.003,1,0.680,0.790,0.543,0.380,better


With all six families tuned, two are clearly better than logistic regression on log-loss: gradient boosting (0.541, interval [-0.028, -0.007], F1 0.748) and the random forest (0.543, interval [-0.027, -0.003], F1 0.751). Neither changes F1 clearly. LDA, KNN and the decision tree show no clear difference, and Naive Bayes stays worse.


## Findings

| Family | Default F1 | Tuned F1 | Tuned log-loss | Tuning kept? | Against logistic regression |
|---|---|---|---|---|---|
| Logistic regression | 0.749 | 0.749 | 0.558 | default (reference) | reference |
| Gradient boosting | 0.744 | 0.748 | 0.541 | yes (notebook 3) | better |
| Random forest | 0.745 | 0.751 | 0.543 | yes | better |
| LDA | 0.745 | 0.743 | 0.565 | yes | no clear difference |
| KNN | 0.731 | 0.743 | 0.565 | yes | no clear difference |
| Decision tree | 0.631 | 0.742 | 0.557 | yes | no clear difference |
| Naive Bayes | 0.687 | 0.725 | 0.582 | yes | worse |

- Tuning helped every family, most where the default overfitted or was overconfident. The tree gained +0.111 F1 by shrinking to 4 leaves, KNN and Naive Bayes fixed their probabilities, and the forest gained through larger leaves.
- Judged on log-loss, the two tree ensembles now give better probabilities than logistic regression, but none of the tuned models changes F1 clearly.
- The models reach similar F1 in different ways: a linear boundary, boosted small trees, averaged deep trees and local neighbourhoods. That makes them candidates for blends.


# 5. Scoreboard and model choice

## 5.1. Final scoreboard

Every family enters with its chosen setting and is judged against logistic regression. Finalists are logistic regression and the models with no clear difference from it; the two families that pass the rule (random forest and gradient boosting) take part through the blends and the choice in section 5.5.


In [54]:
board_final = judge({"dummy": reference_runs["dummy"], **tuned_runs}, reference="logistic").sort_values("f1", ascending=False)
board_final.loc["dummy", "n_features"] = 0
display(board_final[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
                     "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

# decision: finalists are the models not clearly worse than logistic regression
finalists = board_final.index[board_final["verdict"].isin(["reference", "no clear difference"])].tolist()
print("Finalists:", finalists)

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
random_forest,41.000,0.751,0.002,-0.009,0.015,-0.014,-0.027,-0.003,1,0.680,0.790,0.543,0.380,better
logistic,20.250,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
hist_gb,41.000,0.748,-0.001,-0.013,0.014,-0.017,-0.028,-0.007,0,0.670,0.790,0.541,0.360,better
lda,20.250,0.743,-0.006,-0.015,0.003,0.007,-0.002,0.015,4,0.659,0.780,0.565,0.330,no clear difference
knn,20.250,0.743,-0.006,-0.019,0.006,0.007,-0.015,0.031,3,0.672,0.780,0.565,0.360,no clear difference
decision_tree,41.000,0.742,-0.007,-0.020,0.008,-0.001,-0.016,0.014,2,0.649,0.759,0.557,0.310,no clear difference
naive_bayes,20.250,0.725,-0.023,-0.039,-0.010,0.024,0.010,0.037,4,0.647,0.756,0.582,0.360,worse
dummy,0.000,0.709,-0.040,-0.058,-0.018,0.133,0.074,0.193,4,0.500,0.500,0.689,0.300,worse


Finalists: ['logistic', 'lda', 'knn', 'decision_tree']


Two tuned models pass the rule against logistic regression: the random forest (log-loss 0.543, F1 0.751) and gradient boosting (log-loss 0.541, F1 0.748). Both rank the purchases better (ROC-AUC 0.790 against 0.785). LDA, KNN and the decision tree are not clearly different and form the finalists with logistic regression; Naive Bayes is worse. Apart from Naive Bayes, all F1 values lie within 0.009 of one another, so the differences show up in the probabilities, not in the labels.


In [55]:
fold_f1_final = pd.DataFrame({name: tuned_runs[name][0]["f1"] for name in finalists})
fold_f1_final["always repeat"] = [f1_floor(part["y"]) for _, part in tuned_runs["logistic"][1].groupby("fold")]

fig = go.Figure()
for name in fold_f1_final:
    floor = name == "always repeat"
    fig.add_trace(go.Scatter(
        x=fold_f1_final.index, y=fold_f1_final[name], mode="lines" if floor else "lines+markers", name=name,
        line=dict(dash="dash", color="grey") if floor else None,
        hovertemplate="%{x}: F1 %{y:.3f}<extra>" + name + "</extra>",
    ))
fig.update_layout(
    title="Finalists: F1 in each validation year (tuned settings)",
    xaxis=dict(title="Validation year", tickmode="array", tickvals=VAL_YEARS),
    yaxis_title="F1 at each model's pooled threshold",
    height=430,
)
fig.write_html(FIGURES_DIR / "04_finalists_f1_by_year.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_finalists_f1_by_year.png", **PNG_OPTIONS)
fig.show()

display(fold_f1_final.round(3))

,logistic,lda,knn,decision_tree,always repeat
fold,,,,,
2020,0.764,0.758,0.758,0.745,0.745
2021,0.762,0.764,0.749,0.755,0.709
2022,0.709,0.690,0.689,0.713,0.649
2023,0.755,0.754,0.769,0.748,0.728


No finalist leads in every year. KNN is the best in 2023 (0.769) but the weakest in 2022 (0.689). Logistic regression is the best or close to it in 2020 and 2021. All four fall to their lowest F1 in 2022, the year with the lowest floor (0.649).


## 5.2. Blends

A blend averages the probabilities of several models (soft voting). It can only help where the members disagree, so their agreement is checked first.

In [56]:
# How similar are the finalists' predictions? Blends gain only where models disagree.
probabilities = pd.DataFrame({name: tuned_runs[name][1]["p"].to_numpy() for name in finalists})
labels = pd.DataFrame({name: tuned_runs[name][1]["p"].to_numpy() >= tuned_runs[name][2] for name in finalists})
agreement = pd.DataFrame(
    [[(labels[a] == labels[b]).mean() for b in finalists] for a in finalists], index=finalists, columns=finalists
)
correlation = probabilities.corr(method="spearman")

fig = go.Figure(go.Heatmap(
    z=correlation.to_numpy(), x=finalists, y=finalists, zmin=-1, zmax=1, colorscale=DIVERGING,
    text=correlation.round(2).to_numpy(), texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}<br>rank correlation %{z:.3f}<extra></extra>",
))
fig.update_layout(title="Finalists: rank correlation of out-of-fold probabilities", height=430)
fig.write_html(FIGURES_DIR / "04_finalists_correlation.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_finalists_correlation.png", **PNG_OPTIONS)
fig.show()

print("Share of rows with the same predicted label:")
display(agreement.round(3))

Share of rows with the same predicted label:


,logistic,lda,knn,decision_tree
logistic,1.000,0.916,0.869,0.822
lda,0.916,1.000,0.860,0.820
knn,0.869,0.860,1.000,0.802
decision_tree,0.822,0.820,0.802,1.000


Logistic regression and LDA agree on 91.6% of the labels, so they carry largely the same information. The decision tree is the most different from every other finalist (80.2–82.2% agreement), and KNN agrees with logistic regression on 86.9%. Blending models that disagree more has more room to help.


In [57]:
def blend(names, weights=None):
    """Soft vote: weighted mean of the members' probabilities, with its own pooled threshold."""
    weights = np.ones(len(names)) / len(names) if weights is None else np.asarray(weights) / np.sum(weights)
    base = tuned_runs[names[0]][1]
    for name in names[1:]:
        assert tuned_runs[name][1]["ID"].equals(base["ID"]), "Members must be scored on the same rows."

    oof = base[["fold", "ID", "customer_id", "seen", "y"]].copy()
    oof["p"] = np.column_stack([tuned_runs[name][1]["p"] for name in names]) @ weights
    threshold, _, _ = best_threshold(oof["y"], oof["p"])

    rows = []
    for year, part in oof.groupby("fold"):
        rows.append({
            "fold": year,
            "n_features": sum(tuned_runs[name][0].loc[year, "n_features"] for name in names),  # total member inputs
            "log_loss": log_loss(part["y"], part["p"], labels=[0, 1]),
            "train_log_loss": np.nan,  # members were fitted separately
            "roc_auc": roc_auc_score(part["y"], part["p"]),
            "fit_seconds": sum(tuned_runs[name][0].loc[year, "fit_seconds"] for name in names),
            "f1": f1_score(part["y"], part["p"] >= threshold),
        })
    return pd.DataFrame(rows).set_index("fold"), oof, threshold


# decision: equal weights only; fitting weights on the same folds would add another layer of selection
blend_members = {
    "logistic + forest": ["logistic", "random_forest"],
    "logistic + boosting": ["logistic", "hist_gb"],
    "logistic + knn": ["logistic", "knn"],
    "logistic + forest + boosting": ["logistic", "random_forest", "hist_gb"],
    "all finalists": finalists,
}
blend_runs = {name: blend(members) for name, members in blend_members.items()}

board_blend = judge({"logistic": tuned_runs["logistic"], **blend_runs}, reference="logistic")
display(board_blend[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
                     "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,
logistic,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
logistic + forest,0.754,0.005,-0.002,0.014,-0.015,-0.024,-0.008,0,0.682,0.792,0.543,0.370,better
logistic + boosting,0.756,0.007,-0.001,0.015,-0.016,-0.023,-0.009,0,0.701,0.792,0.542,0.400,better
logistic + knn,0.748,-0.000,-0.008,0.009,-0.010,-0.019,-0.002,0,0.661,0.787,0.548,0.340,better
logistic + forest + boosting,0.754,0.006,-0.004,0.018,-0.018,-0.027,-0.010,0,0.671,0.793,0.540,0.350,better
all finalists,0.748,-0.000,-0.009,0.011,-0.013,-0.023,-0.004,0,0.657,0.789,0.545,0.330,better


All five blends pass the rule against logistic regression: log-loss falls from 0.558 to 0.540–0.548, every interval lies below zero, and no blend is worse in any year. ROC-AUC rises to 0.787–0.793. F1 moves little (0.748–0.756), with intervals that include zero.

The best probabilities come from logistic regression + forest + boosting (log-loss 0.540, ROC-AUC 0.793) and logistic regression + boosting (log-loss 0.542, F1 0.756). The blend of all four finalists gains least on ranking (0.789), because LDA adds little beyond logistic regression.


## 5.3. Weighted ensembles

The blends above give every member the same weight. Two ways of learning the weights are tested on four members (logistic regression, tuned gradient boosting, random forest and KNN):

- **Ensemble selection (hill-climbing):** start with no models and repeatedly add, with replacement, the member that lowers log-loss most. A member added three times gets three votes. Weights are never negative and stay coarse, which limits overfitting.
- **Stacking:** a logistic regression learns how to combine the members' log-odds. It is regularised (C = 0.1) because it learns from a few thousand rows.

Weights learned on the same rows they are scored on would look better than they are. Here each validation year is combined with weights learned on the earlier years only: 2021 with weights from 2020, 2022 from 2020–2021, and 2023 from 2020–2022. 2020 has no earlier year, so it uses equal weights. The resulting predictions are judged with the usual rule against logistic regression and against the equal-weight blend of the same members.


In [58]:
ENSEMBLE_MEMBERS = ["logistic", "hist_gb", "random_forest", "knn"]
base_oof = tuned_runs[ENSEMBLE_MEMBERS[0]][1]
for name in ENSEMBLE_MEMBERS[1:]:
    assert tuned_runs[name][1]["ID"].equals(base_oof["ID"]), "Members must be scored on the same rows."
P_oof = np.column_stack([tuned_runs[name][1]["p"] for name in ENSEMBLE_MEMBERS])
y_oof = base_oof["y"].to_numpy()
fold_oof = base_oof["fold"].to_numpy()


def logit(P):
    P = np.clip(P, 1e-6, 1 - 1e-6)
    return np.log(P / (1 - P))


def hill_climb(P, y, rounds=50):
    """Ensemble selection with replacement; returns weights that sum to 1 at the best round."""
    counts, best_counts, best_loss = np.zeros(P.shape[1]), None, np.inf
    total = np.zeros(len(y))
    for _ in range(rounds):
        losses = [log_loss(y, (total + P[:, j]) / (counts.sum() + 1), labels=[0, 1]) for j in range(P.shape[1])]
        j = int(np.argmin(losses))
        counts[j] += 1
        total += P[:, j]
        if losses[j] < best_loss - 1e-6:
            best_loss, best_counts = losses[j], counts.copy()
    return best_counts / best_counts.sum()


def fit_stack(P, y):
    return LogisticRegression(C=0.1, max_iter=5000).fit(logit(P), y)


COMBINERS = {}  # fitted combiners for the final model, keyed by ensemble name


def chronological_ensemble(kind):
    """Each year combined with weights learned on earlier years only; 2020 uses equal weights."""
    p = P_oof.mean(axis=1)
    for year in VAL_YEARS[1:]:
        past, now = fold_oof < year, fold_oof == year
        if kind == "hill_climb":
            p[now] = P_oof[now] @ hill_climb(P_oof[past], y_oof[past])
        else:
            p[now] = fit_stack(P_oof[past], y_oof[past]).predict_proba(logit(P_oof[now]))[:, 1]
    return p


def run_from_oof(p, n_features):
    """Same output as score_folds for a combined prediction."""
    oof = base_oof[["fold", "ID", "customer_id", "seen", "y"]].copy()
    oof["p"] = p
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    rows = [{"fold": year, "n_features": n_features,
             "log_loss": log_loss(part["y"], part["p"], labels=[0, 1]), "train_log_loss": np.nan,
             "roc_auc": roc_auc_score(part["y"], part["p"]), "fit_seconds": np.nan,
             "f1": f1_score(part["y"], part["p"] >= threshold)} for year, part in oof.groupby("fold")]
    return pd.DataFrame(rows).set_index("fold"), oof, threshold


n_members = sum(tuned_runs[name][0]["n_features"].mean() for name in ENSEMBLE_MEMBERS)
weighted_runs = {
    "equal weights (4)": blend(ENSEMBLE_MEMBERS),
    "hill-climbing (4)": run_from_oof(chronological_ensemble("hill_climb"), n_members),
    "stacking (4)": run_from_oof(chronological_ensemble("stack"), n_members),
}

# Weights for the final model come from all four development years
final_weights = hill_climb(P_oof, y_oof)
final_stack = fit_stack(P_oof, y_oof)
COMBINERS["hill-climbing (4)"] = lambda P: P @ final_weights
COMBINERS["stacking (4)"] = lambda P: final_stack.predict_proba(logit(P))[:, 1]
print("Hill-climbing weights (all years):", dict(zip(ENSEMBLE_MEMBERS, final_weights.round(3))))
print("Stacking coefficients on log-odds:", dict(zip(ENSEMBLE_MEMBERS, final_stack.coef_[0].round(3))))

cols = ["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
        "roc_auc", "log_loss", "threshold", "verdict"]
print("\nAgainst logistic regression")
display(judge({"logistic": tuned_runs["logistic"], **weighted_runs}, reference="logistic")[cols])
print("Against the equal-weight blend of the same members")
display(judge(weighted_runs, reference="equal weights (4)")[cols])

# The weighted ensembles join the blends for the model choice
for name, run in weighted_runs.items():
    blend_runs[name] = run
    blend_members[name] = ENSEMBLE_MEMBERS
board_blend = judge({"logistic": tuned_runs["logistic"], **blend_runs}, reference="logistic")


Hill-climbing weights (all years): {'logistic': np.float64(0.154), 'hist_gb': np.float64(0.538), 'random_forest': np.float64(0.308), 'knn': np.float64(0.0)}
Stacking coefficients on log-odds: {'logistic': np.float64(0.071), 'hist_gb': np.float64(0.501), 'random_forest': np.float64(0.382), 'knn': np.float64(0.072)}

Against logistic regression


,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,
logistic,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.785,0.558,0.390,reference
equal weights (4),0.754,0.005,-0.004,0.018,-0.017,-0.027,-0.008,0,0.793,0.541,0.350,better
hill-climbing (4),0.747,-0.002,-0.013,0.009,-0.015,-0.026,-0.006,0,0.791,0.542,0.380,better
stacking (4),0.749,0.001,-0.009,0.013,-0.011,-0.023,-0.002,1,0.792,0.546,0.390,better


Against the equal-weight blend of the same members


,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,
equal weights (4),0.754,0.000,NaN,NaN,0.000,NaN,NaN,0,0.793,0.541,0.350,reference
hill-climbing (4),0.747,-0.007,-0.019,0.001,0.002,-0.000,0.004,2,0.791,0.542,0.380,no clear difference
stacking (4),0.749,-0.005,-0.013,0.002,0.006,0.003,0.009,3,0.792,0.546,0.390,worse


**Findings**

- All three four-model ensembles pass the rule against logistic regression. The equal-weight blend gives log-loss 0.541 and F1 0.754, hill-climbing 0.542 and 0.747, and stacking 0.546 and 0.749.
- Learned weights do not beat equal weights. Against the equal-weight blend of the same members, hill-climbing shows no clear difference (log-loss +0.002) and stacking is worse (log-loss +0.006, interval [0.003, 0.009]).
- On all four years, hill-climbing gives gradient boosting the largest weight (0.54), then the random forest (0.31) and logistic regression (0.15), and drops KNN. Stacking ranks the members the same way. The weights shift from year to year, which is why fixing them on earlier years costs accuracy.
- With four members of similar quality, averaging is already close to the best combination; learning weights adds variance without adding information.


## 5.4. Time-respecting nested check

The tuned scores above come from searches that saw all four validation years and then were scored on the same years, so they carry some optimism. Here each family is tuned again using only the years before each validation year, and then scored on that year once. The threshold also comes from the earlier years only. The question is whether tuning still beats the default, and still matches logistic regression, when the search cannot see the year it is scored on. 2020 has no earlier year, so the outer years are 2021–2023.

In [59]:
NESTED_TRIALS = 20  # per outer year and family; smaller than the main searches to keep the run time reasonable


class FoldSubset:
    """The same cached matrices, restricted to some validation years."""

    def __init__(self, cache, positions):
        self.folds = [cache.folds[i] for i in positions]


def suggest_hgb(trial):
    # Same search space as notebook 3
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_iter": trial.suggest_int("max_iter", 50, 800, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 4, 64, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 300, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 30, log=True),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
    }


def make_hgb(params):
    return HistGradientBoostingClassifier(
        **params, categorical_features=list(range(len(branch_roles("tree")["low"]))),
        early_stopping=False, random_state=RANDOM_STATE,
    )


# family: (cache, suggest, make, default model)
nested_families = {
    "hist_gb": (tree_cache, suggest_hgb, make_hgb, make_hgb({})),
    "random_forest": (tree_cache, suggest_forest, make_forest,
                      RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    "knn": (lambda params: knn_caches[params["columns"]], suggest_knn, make_knn, KNeighborsClassifier()),
}


def outer_f1(cache, model, inner, outer):
    """Threshold from the inner years' predictions, F1 on the outer year."""
    _, _, threshold = evaluate_cached(FoldSubset(cache, inner), model)
    _, oof, _ = evaluate_cached(FoldSubset(cache, [outer]), model)
    return f1_score(oof["y"], oof["p"] >= threshold)


nested_rows = []
for outer in range(1, len(VAL_YEARS)):  # 2020 has no earlier year to tune on
    inner = list(range(outer))
    logistic_f1 = outer_f1(nb_cache, LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), inner, outer)

    for family, (cache, suggest, make, default_model) in nested_families.items():
        def objective(trial):
            params = suggest(trial)
            table, oof, threshold = evaluate_cached(FoldSubset(pick_cache(cache, params), inner), make(params))
            return f1_score(oof["y"], oof["p"] >= threshold)

        # Tuned on the earlier years only; the outer year is never seen by the search
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objective, n_trials=NESTED_TRIALS, show_progress_bar=False)
        best = study.best_params

        default_cache = cache if not callable(cache) else knn_caches["selected"]
        nested_rows.append({
            "outer_year": VAL_YEARS[outer],
            "family": family,
            "inner_years": len(inner),
            "nested_tuned": outer_f1(pick_cache(cache, best), make(best), inner, outer),
            "default": outer_f1(default_cache, default_model, inner, outer),
            "logistic": logistic_f1,
        })
        print(f"{VAL_YEARS[outer]} {family}: done")

nested = pd.DataFrame(nested_rows)
nested["gain_vs_default"] = nested["nested_tuned"] - nested["default"]
nested["gain_vs_logistic"] = nested["nested_tuned"] - nested["logistic"]

2021 hist_gb: done
2021 random_forest: done
2021 knn: done
2022 hist_gb: done
2022 random_forest: done
2022 knn: done
2023 hist_gb: done
2023 random_forest: done
2023 knn: done


In [60]:
display(nested.round(3))

,outer_year,family,inner_years,nested_tuned,default,logistic,gain_vs_default,gain_vs_logistic
0,2021,hist_gb,1,0.729,0.748,0.765,-0.019,-0.036
1,2021,random_forest,1,0.768,0.750,0.765,0.018,0.003
2,2021,knn,1,0.740,0.722,0.765,0.018,-0.026
3,2022,hist_gb,2,0.688,0.705,0.711,-0.017,-0.023
4,2022,random_forest,2,0.708,0.701,0.711,0.007,-0.003
5,2022,knn,2,0.721,0.696,0.711,0.025,0.010
6,2023,hist_gb,3,0.763,0.764,0.742,-0.002,0.020
7,2023,random_forest,3,0.765,0.753,0.742,0.012,0.022
8,2023,knn,3,0.763,0.748,0.742,0.015,0.020


In [61]:
display(nested.groupby("family")[["nested_tuned", "default", "logistic", "gain_vs_default", "gain_vs_logistic"]]
        .mean().round(3))

,nested_tuned,default,logistic,gain_vs_default,gain_vs_logistic
family,,,,,
hist_gb,0.726,0.739,0.739,-0.012,-0.013
knn,0.741,0.722,0.739,0.019,0.002
random_forest,0.747,0.735,0.739,0.012,0.007


When tuned only on earlier years, the random forest keeps its gain: +0.012 F1 over its default on average and +0.007 over logistic regression, ahead in 2021 and 2023 and slightly behind in 2022. KNN gains +0.019 over its default and ends level with logistic regression (+0.002). Gradient boosting loses its advantage in this check: tuned on one or two years only, it falls below its default (-0.012) and below logistic regression (-0.013), recovering only in 2023 (+0.020), when three years are available. Its tuning needs more history than the early outer years provide.


## 5.5. Choice

The rule keeps logistic regression unless an alternative passes the improvement test against it. Ten alternatives pass: the random forest, gradient boosting, the five blends and the three four-model ensembles. Among passing options, the one with the lowest log-loss is chosen; the number of inputs only breaks ties, because a blend's count is the sum of its members' counts and is not comparable with a single model's.

The chosen model is the equal-weight blend of logistic regression, the random forest and gradient boosting, with threshold 0.35. It has the lowest log-loss of all options (0.540, interval of the difference [-0.027, -0.010]) and F1 0.754. The runner-up is the equal-weight blend of the four ensemble members (log-loss 0.541, F1 0.754).

This ranking replaced an earlier version that ordered passing options by input count, which had selected logistic regression + KNN. The change was made after that run's holdout results had been seen, so the holdout is not fully untouched for this choice.


In [62]:
# decision: logistic regression stays unless a tuned model or blend passes the rule against it
all_runs = {**tuned_runs, **blend_runs}
candidates = pd.concat([board_final.drop(index=["dummy", "logistic"]), board_blend.drop(index="logistic")])
better = candidates.index[candidates["verdict"] == "better"].tolist()
# Among passing options: lowest log-loss, the main criterion (a blend's input count is a sum, so not comparable)
chosen_model = candidates.loc[better].sort_values(["ll_diff", "n_features"]).index[0] if better else "logistic"
threshold = all_runs[chosen_model][2]
# Runner-up: the blend with the lowest log-loss that is not the chosen model
runner_up = board_blend.drop(index=["logistic", chosen_model], errors="ignore").sort_values("ll_diff").index[0]
members_of = lambda name: blend_members.get(name, [name])

print(f"Passing against logistic regression: {better or 'none'}")
print(f"Chosen: {chosen_model} (threshold {threshold:.2f}) | runner-up: {runner_up} {members_of(runner_up)}")

log.record(
    "D41", "default models", "default models against logistic regression",
    f"Verdicts: {board_default['verdict'].to_dict()}.",
    0, 0, carries={"default_f1": board_default["f1"].round(4).to_dict(),
                   "default_log_loss": board_default["log_loss"].round(4).to_dict()},
)
log.record(
    "D42", "tuning", "each family tuned with two searches and kept only if it passed against its default",
    "Chosen settings per family recorded in carries.",
    0, 0, carries={"naive_bayes": nb_params, "lda": lda_params, "knn": knn_params, "decision_tree": tree_params,
                   "random_forest": forest_params, "tuned_f1": board_final["f1"].round(4).to_dict()},
)
log.record(
    "D43", "final model", f"{chosen_model}",
    f"Passing against logistic regression: {better or 'none'}. Finalists {finalists}. Nested gains over logistic "
    f"regression: {nested.groupby('family')['gain_vs_logistic'].mean().round(3).to_dict()}. "
    f"Runner-up: {runner_up} (F1 {board_blend.loc[runner_up, 'f1']:.3f}).",
    0, 0, carries={"chosen": chosen_model, "threshold": float(threshold), "runner_up": members_of(runner_up)},
)
display(log.to_frame().tail(3))


Passing against logistic regression: ['random_forest', 'hist_gb', 'logistic + forest', 'logistic + boosting', 'logistic + knn', 'logistic + forest + boosting', 'all finalists', 'equal weights (4)', 'hill-climbing (4)', 'stacking (4)']
Chosen: logistic + forest + boosting (threshold 0.35) | runner-up: equal weights (4) ['logistic', 'hist_gb', 'random_forest', 'knn']


,step,column,action,reason,rows_train,rows_test
39,D41,default models,default models against logistic regression,"Verdicts: {'logistic': 'reference', 'lda_selected': 'worse', 'random_forest': 'no clear differen...",0,0
40,D42,tuning,each family tuned with two searches and kept only if it passed against its default,Chosen settings per family recorded in carries.,0,0
41,D43,final model,logistic + forest + boosting,"Passing against logistic regression: ['random_forest', 'hist_gb', 'logistic + forest', 'logistic...",0,0


## Findings

| Candidate | F1 | F1 difference [95% interval] | Log-loss difference [95% interval] | ROC-AUC | Verdict |
|---|---|---|---|---|---|
| Logistic regression | 0.749 | reference | reference | 0.785 | reference |
| Random forest (tuned) | 0.751 | +0.002 [-0.009, 0.015] | -0.014 [-0.027, -0.003] | 0.790 | better |
| Gradient boosting (tuned) | 0.748 | -0.001 [-0.013, 0.014] | -0.017 [-0.028, -0.007] | 0.790 | better |
| Logistic + forest + boosting | 0.754 | +0.006 [-0.004, 0.018] | -0.018 [-0.027, -0.010] | 0.793 | better, chosen |
| Equal weights (4) | 0.754 | +0.005 [-0.004, 0.018] | -0.017 [-0.027, -0.008] | 0.793 | better, runner-up |

- The tree ensembles and their blends give clearly better probabilities than logistic regression, while F1 changes by at most +0.007, with intervals that include zero.
- Combining a linear model with two tree ensembles gives the best probabilities: the members make different errors, and averaging cancels part of them.
- Learned weights did not improve on equal weights.
- The forest's tuning gain survives the nested check; gradient boosting's depends on having several years to tune on.


# 6. Threshold and calibration

The chosen blend's threshold (0.35) comes from its pooled development predictions, with the same rule as every other model. This section checks whether its predicted probabilities match the observed share of repeats.

## 6.1. Calibration


In [63]:
# Reliability of the chosen model's probabilities, with the runner-up blend for contrast
compare = {f"chosen ({chosen_model})": all_runs[chosen_model][1], f"runner-up ({runner_up})": all_runs[runner_up][1]}

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
                         line=dict(dash="dash", color="grey")))
calibration_rows = []
for name, oof in compare.items():
    observed, predicted = calibration_curve(oof["y"], oof["p"], n_bins=10, strategy="quantile")
    fig.add_trace(go.Scatter(x=predicted, y=observed, mode="lines+markers", name=name,
                             hovertemplate="predicted %{x:.2f}<br>observed %{y:.2f}<extra>" + name + "</extra>"))
    calibration_rows.append({
        "model": name,
        "brier": brier_score_loss(oof["y"], oof["p"]),
        "mean_predicted": oof["p"].mean(),
        "observed_share": oof["y"].mean(),
        "largest_decile_gap": np.abs(observed - predicted).max(),
    })
fig.update_layout(
    title="Predicted against observed repeat share (deciles of the predictions, development years)",
    xaxis=dict(title="Mean predicted probability", range=[0, 1]),
    yaxis=dict(title="Observed repeat share", range=[0, 1]),
    height=480,
)
fig.write_html(FIGURES_DIR / "04_calibration_final.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_calibration_final.png", **PNG_OPTIONS)
fig.show()

display(pd.DataFrame(calibration_rows).set_index("model").round(3))

,brier,mean_predicted,observed_share,largest_decile_gap
model,,,,
chosen (logistic + forest + boosting),0.184,0.546,0.549,0.029
runner-up (equal weights (4)),0.184,0.542,0.549,0.045


The chosen blend is well calibrated: its mean prediction is 0.546 against an observed share of 0.549, its Brier score is 0.184, and no decile is off by more than 0.029. The runner-up has the same Brier score with a larger decile gap (0.045).


## 6.2. Calibration by year

In [64]:
# Calibration year by year: does the model over- or under-predict repeats as the years change?
oof = all_runs[chosen_model][1]
by_year = oof.groupby("fold").agg(
    rows=("y", "size"),
    observed_share=("y", "mean"),
    mean_predicted=("p", "mean"),
    predicted_positive_share=("p", lambda p: (p >= threshold).mean()),
)
by_year["prediction_gap"] = by_year["mean_predicted"] - by_year["observed_share"]
display(by_year.round(3))

,rows,observed_share,mean_predicted,predicted_positive_share,prediction_gap
fold,,,,,
2020,670,0.594,0.569,0.737,-0.025
2021,782,0.549,0.537,0.670,-0.012
2022,628,0.481,0.553,0.720,0.073
2023,536,0.573,0.522,0.728,-0.051


The mean prediction moves less between years (0.522–0.569) than the observed share of repeats (0.481–0.594). The blend underpredicts in 2020, 2021 and 2023 by 0.012–0.051 and overpredicts in 2022 by 0.073, the year with the fewest repeats. At the fixed threshold it flags 67–74% of purchases as repeats every year, so its errors follow the yearly level of repeats rather than the individual customers.


## Findings

- The chosen blend's threshold is 0.35, from its pooled development predictions.
- Probabilities are close to calibrated overall (Brier 0.184, largest decile gap 0.029).
- The share of repeats changes from year to year by up to 0.11, and the model does not track that level. This drift sets a limit on F1 that tuning cannot remove.


# 7. Holdout assessment

The holdout (purchases from 2024-01-01) has not been used for any choice so far. Every model is refitted on the development years and scored once, with the settings and threshold already fixed. The chosen model's result is its unbiased estimate; the other finalists are recorded for reference only.

In [65]:
def family_pipe(name):
    """Complete pipeline of one family with its chosen settings (defaults when tuning was not kept)."""
    linear = lambda model, select=True: (build_linear_pipeline(model, select=select), linear_inputs)
    tree = lambda model: (build_tree_pipeline(model), tree_inputs)
    if name == "logistic":
        return linear(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    if name == "hist_gb":
        return tree(HistGradientBoostingClassifier(**hgb_params, random_state=RANDOM_STATE))
    if name == "naive_bayes":
        return linear(make_nb(nb_params) if nb_params else GaussianNB())
    if name == "lda":
        return linear(make_lda(lda_params) if lda_params else LinearDiscriminantAnalysis(),
                      select=lda_params.get("columns", "selected") == "selected")
    if name == "knn":
        return linear(make_knn(knn_params) if knn_params else KNeighborsClassifier(),
                      select=knn_params.get("columns", "selected") == "selected")
    if name == "decision_tree":
        return tree(make_tree(tree_params) if tree_params else DecisionTreeClassifier(random_state=RANDOM_STATE))
    if name == "random_forest":
        return tree(make_forest(forest_params) if forest_params
                    else RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
    raise ValueError(name)


def fit_members(fit_rows, names):
    return {name: clone(final_pipes[name][0]).fit(fit_rows[final_pipes[name][1]], fit_rows[target]) for name in names}


def predict_members(fitted, rows, ensemble=None):
    """Probability of a single model, an equal-weight blend, or a weighted ensemble (fitted combiner)."""
    P = np.column_stack([model.predict_proba(rows[final_pipes[name][1]])[:, 1] for name, model in fitted.items()])
    return COMBINERS[ensemble](P) if ensemble in COMBINERS else P.mean(axis=1)


# Every finalist, the chosen model and the runner-up; blends are scored from their members
shown = list(dict.fromkeys(finalists + [chosen_model, runner_up]))
families = list(dict.fromkeys(m for name in shown for m in members_of(name)))
final_pipes = {name: family_pipe(name) for name in families}
fitted_dev = fit_members(dev, families)
p_holdout = pd.DataFrame({name: predict_members({m: fitted_dev[m] for m in members_of(name)}, holdout, name)
                          for name in shown})

# Thresholds stay those chosen on the development years
thresholds = {name: all_runs[name][2] for name in shown}
y_holdout = holdout[target].to_numpy()
seen = holdout["customer_id"].isin(dev["customer_id"]).to_numpy()

holdout_rows = []
for name, p in p_holdout.items():
    predicted = p.to_numpy() >= thresholds[name]
    holdout_rows.append({
        "model": name,
        "threshold": thresholds[name],
        "f1": f1_score(y_holdout, predicted),
        "f1_seen": f1_score(y_holdout[seen], predicted[seen]),
        "f1_unseen": f1_score(y_holdout[~seen], predicted[~seen]),
        "balanced_accuracy": balanced_accuracy_score(y_holdout, predicted),
        "roc_auc": roc_auc_score(y_holdout, p),
        "log_loss": log_loss(y_holdout, p),
        "predicted_positive_share": predicted.mean(),
    })
board_holdout = pd.DataFrame(holdout_rows).set_index("model")
board_holdout.insert(2, "floor", f1_floor(y_holdout))

print(f"Holdout: {len(holdout)} purchases, {holdout['customer_id'].nunique()} customers, "
      f"{seen.sum()} rows from customers seen in development, repeat share {y_holdout.mean():.3f}")
display(board_holdout.round(3))


Holdout: 835 purchases, 215 customers, 757 rows from customers seen in development, repeat share 0.605


,threshold,f1,floor,f1_seen,f1_unseen,balanced_accuracy,roc_auc,log_loss,predicted_positive_share
model,,,,,,,,,
logistic,0.390,0.780,0.754,0.793,0.576,0.678,0.782,0.553,0.703
lda,0.330,0.785,0.754,0.798,0.567,0.679,0.776,0.569,0.716
knn,0.360,0.783,0.754,0.796,0.595,0.677,0.784,0.560,0.714
decision_tree,0.310,0.765,0.754,0.779,0.554,0.663,0.744,0.580,0.691
logistic + forest + boosting,0.350,0.788,0.754,0.803,0.579,0.667,0.796,0.540,0.751
equal weights (4),0.350,0.788,0.754,0.802,0.583,0.672,0.796,0.542,0.741


In [66]:
log.record(
    "D44", "holdout assessment",
    f"{chosen_model} at fixed threshold {threshold:.2f}; finalists recorded only",
    "Settings and thresholds fixed on the development years; holdout results did not change the choice. "
    f"F1 by model: {board_holdout['f1'].round(3).to_dict()}.",
    0, len(holdout), carries={"holdout": board_holdout.round(4).to_dict(orient="index")},
)

**Findings**

- The holdout has 835 purchases from 215 customers. 757 of those rows come from customers already seen in development, and 60.5% of the purchases were repeated.
- The chosen blend scores F1 0.788 at its fixed threshold of 0.35, against the predict-all-repeat floor of 0.754 and 0.780 for logistic regression alone. Its log-loss (0.540) and ROC-AUC (0.796) are also the best on the board.
- The runner-up four-model blend scores the same F1 (0.788). LDA (0.785) and KNN (0.783) are close behind; the decision tree is lowest (0.765).
- New customers remain the weak point. The blend reaches F1 0.579 on unseen customers against 0.803 on seen ones; KNN does best there (0.595).
- The blend predicts a repeat for 75.1% of the holdout, against an actual share of 60.5%. F1 rewards recall, so the threshold leans towards predicting repeats.


# 8. Final fit and submission

The chosen pipeline is refitted on every labelled purchase (development, gap and holdout), keeping its settings and threshold. Before predicting, a check confirms that no training outcome window reaches the test period.

In [67]:
# Refit on every labelled purchase (development, gap and holdout); settings and threshold stay fixed
final_model = fit_members(train, members_of(chosen_model))

assert (train["purchase_date"] + pd.Timedelta(days=GAP_DAYS)).max() < test["purchase_date"].min(), \
    "A training outcome window reaches the test period."

p_test = predict_members(final_model, test, chosen_model)
submission = sample_submission[["ID"]].merge(
    pd.DataFrame({"ID": test["ID"], target: (p_test >= threshold).astype(int)}),
    on="ID", how="left", validate="one_to_one",
)
assert len(submission) == len(test) and submission[target].isin([0, 1]).all()

print(f"Final model ({chosen_model}) fitted on {len(train):,} purchases; threshold {threshold:.2f}")
print(f"Predicted repeat share: test {submission[target].mean():.3f} | holdout "
      f"{board_holdout.loc[chosen_model, 'predicted_positive_share']:.3f}")
print(f"Mean probability: test {p_test.mean():.3f} | holdout {p_holdout[chosen_model].mean():.3f}")


Final model (logistic + forest + boosting) fitted on 6,469 purchases; threshold 0.35
Predicted repeat share: test 0.789 | holdout 0.751
Mean probability: test 0.586 | holdout 0.557


In [68]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

submission_bytes = submission.to_csv(index=False, lineterminator="\n").encode("utf-8")
run_id = hashlib.sha256(submission_bytes).hexdigest()[:16]
stem = f"submission_project_{run_id}"

metadata = {
    "experiment_version": EXPERIMENT_VERSION,
    "input_fingerprint": INPUT_FINGERPRINT,
    "model": chosen_model,
    "threshold": float(threshold),
    "development_f1": float(f1_score(all_runs[chosen_model][1]["y"], all_runs[chosen_model][1]["p"] >= threshold)),
    "holdout_f1": float(board_holdout.loc[chosen_model, "f1"]),
    "predicted_repeat_share": float(submission[target].mean()),
    "training_rows": int(len(train)),
    "members": members_of(chosen_model),
    "runner_up": members_of(runner_up),
    "random_state": int(RANDOM_STATE),
    "package_versions": {pkg: version(pkg) for pkg in ["numpy", "pandas", "scikit-learn", "optuna", "plotly"]},
}

(SUBMISSION_DIR / f"{stem}.csv").write_bytes(submission_bytes)
(SUBMISSION_DIR / f"{stem}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
# The fitted pipelines (one per member) need src/ on the path to load (custom frequency encoder)
joblib.dump(final_model, MODELS_DIR / f"{stem}.joblib")

# Append to the submission log; earlier rows and manually added public scores are kept
log_path = SUBMISSION_DIR / "submission_log.csv"
submission_log = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()
if f"{stem}.csv" not in submission_log.get("file", pd.Series(dtype=str)).tolist():
    submission_log = pd.concat([submission_log, pd.DataFrame([{
        "file": f"{stem}.csv", "created": pd.Timestamp.now(tz="UTC").isoformat(), "model": chosen_model,
        "threshold": threshold, "fold_f1": metadata["development_f1"], "holdout_f1": metadata["holdout_f1"],
        "predicted_repeat_share": metadata["predicted_repeat_share"], "public_score": np.nan, "notes": "notebook 04",
        "run_id": run_id, "metadata_file": f"{stem}.json",
    }])], ignore_index=True)
    submission_log.to_csv(log_path, index=False)

log.record(
    "D45", "submission", f"{stem}.csv",
    f"{chosen_model} refitted on {len(train):,} labelled purchases; threshold {threshold:.2f}; "
    f"{len(submission)} test predictions in sample-submission order.",
    len(train), len(test), carries={"run_id": run_id, "model_file": f"{stem}.joblib"},
)
log.to_json(PROCESSED_DIR / "decision_log_modelling.json")

# Agreement with the first submission (notebook 3, before the notebook 2 review)
previous = pd.read_csv(SUBMISSION_DIR / "ML_Group_07_Version01.csv").set_index("ID").loc[submission["ID"], target]
print(f"Saved {stem}.csv, .json and .joblib")
print(f"Same decision as the first submission on {(previous.to_numpy() == submission[target].to_numpy()).mean():.1%} "
      f"of test rows; its repeat share was {previous.mean():.3f}")
display(submission_log.tail(3))

Saved submission_project_4476b21eaf8bd4f6.csv, .json and .joblib
Same decision as the first submission on 91.3% of test rows; its repeat share was 0.822


,file,created,model,threshold,fold_f1,holdout_f1,predicted_repeat_share,public_score,notes,run_id,metadata_file
4,submission_project_bcffc9f805db7eb7.csv,2026-11-19T00:36:57.862492+00:00,logistic + knn,0.340,0.748,0.780,0.807,NaN,notebook 04,bcffc9f805db7eb7,submission_project_bcffc9f805db7eb7.json
5,submission_homework_c884555b3b3b70df.csv,2026-11-19T01:36:23.491045+00:00,blend_50_50,0.390,0.752,0.786,0.759,NaN,NaN,c884555b3b3b70df2c2c08e034a99428088203c7d4ed17261782db89bc80a9d6,submission_homework_c884555b3b3b70df.json
6,submission_project_4476b21eaf8bd4f6.csv,2026-11-19T01:44:58.304447+00:00,logistic + forest + boosting,0.350,0.754,0.788,0.789,NaN,notebook 04,4476b21eaf8bd4f6,submission_project_4476b21eaf8bd4f6.json


**Findings**

- The final blend is refitted on 6,469 labelled purchases; each of its three members is refitted separately. No training outcome window reaches the test period.
- On the test set the blend predicts a repeat for 78.9% of purchases, against 75.1% on the holdout; the mean probability rises from 0.557 to 0.586.
- The submission agrees with the first submission (logistic regression before the notebook 2 review, 82.2% predicted repeats) on 91.3% of the test rows.
- Public leaderboard: 0.80668 for this submission, against 0.78672 for the first one and 0.79905 for logistic regression + KNN (recorded manually).


# 9. Drift and refit check

The final model predicts a repeat for a larger share of test purchases than of holdout purchases. Two explanations are checked: a change in the inputs between the holdout and the test period, and the effect of refitting on every labelled purchase.

In [69]:
# Numeric inputs used by the models; the noise column is already excluded
numeric_cols = [c for c in linear_inputs if pd.api.types.is_numeric_dtype(train[c])]
last_dev_year = dev[dev["purchase_date"].dt.year == VAL_YEARS[-1]]


def psi(expected, actual, bins=10):
    """Population stability index; bins are deciles of the expected sample."""
    expected, actual = expected.dropna().astype(float), actual.dropna().astype(float)
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-4  # small floor avoids log(0)
    a = np.histogram(actual, edges)[0] / len(actual) + 1e-4
    return float(((a - e) * np.log(a / e)).sum())


psi_table = pd.DataFrame({
    f"{VAL_YEARS[-1]} vs holdout": {c: psi(last_dev_year[c], holdout[c]) for c in numeric_cols},
    "holdout vs test": {c: psi(holdout[c], test[c]) for c in numeric_cols},
}).sort_values("holdout vs test", ascending=False)

# Adversarial check: can a model tell holdout rows from test rows?
adv_X = pd.concat([holdout[numeric_cols], test[numeric_cols]]).astype(float)
adv_y = np.r_[np.zeros(len(holdout)), np.ones(len(test))]
adv_model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE)
adv_p = cross_val_predict(adv_model, adv_X, adv_y, method="predict_proba",
                          cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))[:, 1]
adv_importance = pd.Series(
    permutation_importance(adv_model.fit(adv_X, adv_y), adv_X, adv_y, scoring="roc_auc",
                           n_repeats=3, random_state=RANDOM_STATE).importances_mean,
    index=numeric_cols,
).sort_values(ascending=False)

seen_test = test["customer_id"].isin(train["customer_id"])
print(f"Test: {len(test)} purchases, {test['customer_id'].nunique()} customers, "
      f"{seen_test.mean():.1%} of rows from customers in the training data")
print(f"Adversarial ROC-AUC, holdout vs test: {roc_auc_score(adv_y, adv_p):.3f}")
print("Features that separate them most:", adv_importance.head(3).round(3).to_dict())
display(psi_table.head(10).round(3))

top = psi_table.head(10).iloc[::-1]
fig = go.Figure([go.Bar(y=top.index, x=top[col], orientation="h", name=col, marker_color=SERIES_COLOURS[i])
                 for i, col in enumerate(top.columns)])
fig.add_vline(x=0.1, line_dash="dot", line_color=ACCENT["near_black"])
fig.add_vline(x=0.25, line_dash="dash", line_color=ACCENT["near_black"])
fig.update_layout(title="Population stability index of the inputs", xaxis_title="PSI (0.1 watch, 0.25 shift)",
                  barmode="group", height=520)
fig.show()
fig.write_html(FIGURES_DIR / "04_psi.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_psi.png", **PNG_OPTIONS)

Test: 986 purchases, 206 customers, 94.5% of rows from customers in the training data
Adversarial ROC-AUC, holdout vs test: 0.982
Features that separate them most: {'customer_tenure_days': 0.334, 'customer_since_year': 0.218, 'prior_purchase_count': 0.001}


,2023 vs holdout,holdout vs test
customer_tenure_days,0.448,0.461
prior_std_purchase_value,0.023,0.115
prior_max_purchase_value,0.088,0.088
prior_mean_purchase_value,0.055,0.072
spend_per_year,0.056,0.068
spend_365d,0.102,0.065
prior_purchase_count,0.063,0.062
prior_min_purchase_value,0.033,0.055
Total Desc_ Linha,0.049,0.051
purchases_per_year,0.055,0.036


In [70]:
# Same model and threshold, fitted on development rows only versus on every labelled row
p_test_dev_fit = predict_members({m: fitted_dev[m] for m in members_of(chosen_model)}, test, chosen_model)

refit_check = pd.DataFrame({
    "fitted on": ["development (2015–2023)", "all labelled (2015–2025)"],
    "rows": [len(dev), len(train)],
    "training repeat share": [dev[target].mean(), train[target].mean()],
    "mean probability, test": [p_test_dev_fit.mean(), p_test.mean()],
    "predicted repeat share, test": [(p_test_dev_fit >= threshold).mean(), (p_test >= threshold).mean()],
}).set_index("fitted on")
display(refit_check.round(3))
print(f"Same decision on {((p_test_dev_fit >= threshold) == (p_test >= threshold)).mean():.1%} of test rows")
print("Predicted repeat share of the final model by threshold:",
      {t: round(float((p_test >= t).mean()), 3) for t in sorted({round(threshold, 2), 0.40, 0.45, 0.50})})

,rows,training repeat share,"mean probability, test","predicted repeat share, test"
fitted on,,,,
development (2015–2023),5403,0.551,0.564,0.776
all labelled (2015–2025),6469,0.560,0.586,0.789


Same decision on 96.9% of test rows
Predicted repeat share of the final model by threshold: {0.35: 0.789, 0.4: 0.739, 0.45: 0.672, 0.5: 0.601}


**Findings**

- 94.5% of test rows come from customers present in the training data.
- A classifier still separates holdout and test rows almost perfectly (ROC-AUC 0.982), through `customer_tenure_days` and `customer_since_year`. Both grow with calendar time. Tenure is no longer a direct model input: it enters only through the purchase-process probability and the per-year rates, whose PSI stays below 0.07.
- The refit no longer shifts the predictions much. Fitted on development rows only, the blend predicts a repeat for 77.6% of test purchases; fitted on every labelled row, 78.9%. The two versions agree on 96.9% of test rows.
- Raising the threshold from 0.35 to 0.40 would lower the predicted share from 78.9% to 73.9%. The threshold is kept as chosen on the development years.


# 10. Summary of decisions

| Step | Decision | Evidence |
|---|---|---|
| Validation | Expanding yearly folds 2020–2023 with a 90-day gap; holdout from 2024 | Same design as notebooks 2 and 3, so results are comparable |
| Inputs | Notebook 2 configuration: time-stable rates and the purchase-process probability fitted in each training part | Section 1.4 |
| Threshold | Per model, from pooled out-of-fold predictions (centre of the near-best range) | One rule for every model |
| Comparison rule | Better only if the log-loss interval is below zero, F1 is not clearly worse and log-loss is higher in at most one fold | Log-loss uses every row and detects smaller real differences than F1 |
| Default models | No default model beats logistic regression (F1 0.749, log-loss 0.558) | Section 3 |
| Tuning | Two Optuna searches per family (log-loss and F1), 60 trials each, top 5 rescored; every family kept a tuned setting | Forest log-loss 0.543, boosting 0.541 (notebook 3), both better than logistic regression |
| Blends | Five equal-weight blends, all better than logistic regression | Log-loss 0.540–0.548 |
| Weighted ensembles | Hill-climbing and stacking, weights learned on earlier years | Neither beats equal weights; stacking is worse |
| Nested check | Gains over logistic regression: forest +0.007, KNN +0.002, boosting −0.013 | The forest's gain is robust; boosting needs more years to tune |
| Chosen model | Equal-weight blend of logistic regression, random forest and gradient boosting, threshold 0.35 | Lowest log-loss (0.540) among the options that pass |
| Calibration | Brier score 0.184, largest decile gap 0.029 | Probabilities usable as they are |
| Holdout | F1 0.788 against a floor of 0.754 | Logistic regression alone 0.780 |
| Submission | Refitted on all labelled purchases; public score 0.80668 | Section 8 |
| Drift | Refit agreement 96.9% of test rows | Section 9 |
